# CIAOF Pipeline — Core Integrity Adherence–Outcome Framework

**Purpose.** Construct the disclosure-based EA Governance Index from SEC 10-K filings, add the board TechCom indicator from DEF 14A, merge with the Core Integrity firm-year panel, and estimate H1/H2 with two-way fixed effects and Driscoll–Kraay standard errors, plus Oster bounds and a gated Callaway–Sant'Anna event study.

**Run order.** Upload → Runtime → Run all. Discovery: project-tree-first across csv / parquet / feather / pickle / xlsx / dta; header → value-signature → key-token column resolution; split CI file auto-merged; ticker panels rescued via the SEC map. If the panel cannot be resolved, the run continues in *prefetch-only mode* (inventory printed, firm universe extracted, full text layer fetched and banked). Every cell is cascade-proof: an upstream failure makes downstream cells print a skip banner instead of NameError noise, and the **RUN DIAGNOSTIC** cell at the end always executes and names the root failing stage — paste that block together with the traceback it points to.

**Optional second classifier.** To enable the LLM cross-check of TechCom evidence, add a Colab secret named `ANTHROPIC_API_KEY` (left sidebar → key icon → Add new secret → toggle *Notebook access* on). An API key comes from the Claude Developer Platform console and is billed separately from any Claude.ai subscription. Without the key the notebook runs identically using the deterministic rule classifier only; with it, labels are written once to `llm_techcom_labels.csv` and reused verbatim on every later run.



In [ ]:
# ============ 0 · PARAMETERS ============
# CIAOF-ONLY PIPELINE — the lean branch.
# Carries exactly what the CIAOF paper reports: panel construction, the Core Integrity analyses
# (variance decomposition, primary between/within estimate, inverted-U, H3 allocation, H3b
# directional, H3c quintiles, H3d fit-to-potential), the composite-structure battery, the
# specification curve, and the reconciliation export.
#
# The measurement analyses supporting the separate EA-governance paper — lexicon construction,
# reliability battery, positive control, mandatory-disclosure instruments, event studies — are not
# present and no CIAOF result depends on them. Every analysis block here is VERBATIM from the full
# pipeline, so a result cannot differ between branches.
NB_VERSION = "CIAOF-1.0"
print(f"### CIAOF notebook {NB_VERSION} — if this is not the version you just uploaded, Colab is running a cached file ###")
DRIVE_ROOT = "/content/drive/MyDrive"    # scanned recursively
PROJECT_HINT = "ccaof"                    # LOAD-BEARING: the historical name of the source data
                                          # folder on Drive. Searched first; content inference still
                                          # applies, so a renamed folder is still found — but leave
                                          # this as-is unless the Drive folder is actually renamed.
USER_AGENT = ""                           # normally auto-filled from Colab secrets / Drive cache
YEAR_MIN, YEAR_MAX = 2019, 2025
SLEEP = 0.15                              # ≤10 req/s per SEC fair-access policy
COLMAP = {}                               # optional manual override {canonical: your_column}
CI_FILE_OVERRIDE = ""                     # optional: path to a separate CI-score file
USE_LLM_CODING = True                     # LLM second classifier for TechCom evidence. Labels are
LLM_MODEL = "claude-haiku-4-5-20251001"   # CACHED to Drive keyed by evidence hash and shipped with the
LLM_TEMPERATURE = 0.0                     # replication package, so reruns are bit-identical whether or
MAX_LLM_CALLS = 2000                      # not the API is reachable. Rule classifier remains PRIMARY.
USE_EXTERNAL_CS = True                    # cross-check our Callaway-Santanna estimator against the
                                          # `differences` package. ON by default - a check that never
                                          # runs validates nothing. The COMPARISON is cached to Drive,
                                          # so it installs once and later runs read the stored result.
                                          # It can never alter the primary estimate and never breaks
                                          # the run; set False only for a fully offline session.
SEED = 42                                 # DATED model id (not a moving alias) so the classifier
                                          # version is pinned in the replication package.

LEXICON = [
  "enterprise architecture", "architecture governance", "technology governance",
  "it governance", "architecture review board", "target architecture", "architecture roadmap",
  "technical debt", "application rationalization", "systems consolidation",
  "platform consolidation", "legacy system retirement", "legacy decommissioning",
  "modernization roadmap", "technology standardization", "master data management",
  "data governance", "api-first", "integration platform", "event-driven architecture",
]
PLACEBO = ["competitive advantage", "strategic priorities", "shareholder value", "market leadership"]

print(f"{len(LEXICON)} lexicon terms · {len(PLACEBO)} placebo terms · window {YEAR_MIN}–{YEAR_MAX}")


In [ ]:
# ============ 1 · SETUP — mount, walk, discover; prefetch-only fallback ============
import sys, subprocess
def pipq(*pkgs):
    """Plain install - the form that worked for eleven versions. Two attempted "improvements"
    each broke it: --no-deps starved linearmodels of formulaic, and a numpy/pandas constraints
    file made the dependency set unresolvable so nothing installed at all. Neither was needed:
    the 'everything unreadable' run was never traced to pip, it was only hypothesised.

    What IS kept, because it has evidence behind it: install only what is missing, check the
    module that is ACTUALLY IMPORTED (linearmodels imports fine while linearmodels.panel does
    not), and SHOW pip's error instead of failing silently three cells later.
    """
    import importlib
    TARGET = {"beautifulsoup4": "bs4", "linearmodels": "linearmodels.panel",
              "pyarrow": "pyarrow", "openpyxl": "openpyxl", "lxml": "lxml",
              "tqdm": "tqdm", "differences": "differences"}
    def _ok(pk):
        try:
            importlib.invalidate_caches(); importlib.import_module(TARGET.get(pk, pk)); return True
        except Exception:
            return False
    need = [pk for pk in pkgs if not _ok(pk)]
    if not need:
        return
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need],
                       capture_output=True, text=True)
    still = [pk for pk in need if not _ok(pk)]
    if still:
        print(f"  pip could not make {still} importable (exit {r.returncode}).")
        tail = ((r.stderr or "") + (r.stdout or "")).strip().splitlines()[-4:]
        for ln in tail:
            print("    " + ln[:150])
        print("    → Runtime > Restart session, then Run all. A fresh kernel picks up packages")
        print("      installed in this session; the import cache in a live kernel often will not.")

import os, re, json, time, types, warnings, difflib
import numpy as np, pandas as pd
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")
np.random.seed(42)

def G(n):
    return globals().get(n)

# ---------------------------------------------------------------------------
# TWO-WAY FIXED-EFFECTS OLS - implemented directly, no external estimator package.
# linearmodels repeatedly failed to install in this environment, and every patch around the
# installer was a patch around a dependency we do not need. Everything used from PanelOLS is
# elementary: within transformation, clustered sandwich, Driscoll-Kraay. numpy and pandas are
# preinstalled in Colab, so this removes the single most fragile dependency in the pipeline.
class FEResult:
    def __init__(self, names, beta, cov, nobs, r2w, dfree):
        self.params = pd.Series(beta, index=names)
        self.std_errors = pd.Series(np.sqrt(np.maximum(np.diag(cov), 0)), index=names)
        self.bse = self.std_errors
        with np.errstate(divide="ignore", invalid="ignore"):
            self.tstats = self.params / self.std_errors
        self.tvalues = self.tstats
        from math import erf, sqrt as _sq
        self.pvalues = pd.Series(
            [2 * (1 - 0.5 * (1 + erf(abs(t) / _sq(2)))) if t == t else np.nan for t in self.tstats],
            index=names)
        self.nobs = int(nobs); self.rsquared_within = float(r2w); self.df_resid = int(dfree)
        tab = pd.DataFrame({"coef": self.params.round(6), "std_err": self.std_errors.round(6),
                            "t": self.tstats.round(3), "P>|t|": self.pvalues.round(4)})
        self.summary = types.SimpleNamespace(tables=[None, tab.to_string()])

def _demean2(frame, cols, ent, tim, iters=60, tol=1e-11):
    """Alternating projections: on an unbalanced panel a single firm-then-year pass is NOT the
    two-way within transformation, so iterate to convergence."""
    X = frame[cols].astype(float).copy()
    for _ in range(iters):
        prev = X.to_numpy(copy=True)
        X = X - X.groupby(ent, observed=True).transform("mean")
        X = X - X.groupby(tim, observed=True).transform("mean")
        if np.nanmax(np.abs(X.to_numpy() - prev)) < tol:
            break
    return X

def fe_ols(dep, xs, data, se="clustered", bandwidth=None):
    """Two-way FE regression. data must carry a (entity, time) MultiIndex.
    se='clustered' clusters on entity; se='kernel' is Driscoll-Kraay (Bartlett)."""
    d_ = data.dropna(subset=[dep] + list(xs)).copy()
    ent = pd.Series(d_.index.get_level_values(0), index=d_.index)
    tim = pd.Series(d_.index.get_level_values(1), index=d_.index)
    Z = _demean2(d_, [dep] + list(xs), ent, tim)
    y = Z[dep].to_numpy(float); X = Z[list(xs)].to_numpy(float)
    n, k = X.shape
    XtX_inv = np.linalg.pinv(X.T @ X)
    beta = XtX_inv @ X.T @ y
    e = y - X @ beta
    n_ent = int(ent.nunique()); n_tim = int(tim.nunique())
    dfree = max(1, n - k - n_ent - n_tim + 1)
    if se == "kernel":
        u = X * e[:, None]
        h = pd.DataFrame(u, index=tim.to_numpy()).groupby(level=0).sum().to_numpy()  # cross-sec sum
        T = h.shape[0]
        m = bandwidth if bandwidth is not None else int(np.floor(4 * (T / 100.0) ** (2.0 / 9.0)))
        m = max(0, min(m, T - 1))
        S = h.T @ h
        for j in range(1, m + 1):
            w = 1.0 - j / (m + 1.0)
            G_ = h[j:].T @ h[:-j]
            S = S + w * (G_ + G_.T)
        cov = XtX_inv @ S @ XtX_inv
    else:
        meat = np.zeros((k, k))
        codes = ent.to_numpy()
        for g_ in pd.unique(codes):
            msk = codes == g_
            sg = X[msk].T @ e[msk]
            meat += np.outer(sg, sg)
        cov = XtX_inv @ meat @ XtX_inv
        nc = max(1, len(pd.unique(codes)))
        cov *= (nc / max(1, nc - 1)) * ((n - 1) / max(1, dfree))
    tss = float(((y - y.mean()) ** 2).sum())
    r2w = 1 - float(e @ e) / tss if tss > 0 else np.nan
    return FEResult(list(xs), beta, cov, n, r2w, dfree)


from google.colab import drive

def _mount_drive(path="/content/drive", tries=4):
    """Colab's Drive mount fails transiently with 'credential propagation was unsuccessful'.
    That is an ENVIRONMENT fault, not a data fault, and it is usually cleared by a retry.
    Escalate: plain retry, then unmount-and-remount, then a forced remount. Fall through
    immediately if the mount point is already usable, so a rerun costs nothing."""
    if os.path.isdir(os.path.join(path, "MyDrive")):
        print("  Drive already mounted - reusing")
        return True
    last = None
    for a in range(tries):
        try:
            drive.mount(path, force_remount=(a >= 2))
            if os.path.isdir(os.path.join(path, "MyDrive")):
                print(f"  Drive mounted on attempt {a + 1}")
                return True
        except Exception as e:
            last = e
            print(f"  mount attempt {a + 1}/{tries} failed: {type(e).__name__}: {str(e)[:90]}")
            if a == 1:
                try:
                    drive.flush_and_unmount()
                    print("  unmounted stale mount point; retrying")
                except Exception:
                    pass
        time.sleep(3 * (a + 1))
    print("\n  DRIVE MOUNT FAILED after", tries, "attempts.")
    print("  This is a Colab environment fault. Nothing is wrong with the data or this notebook.")
    print("  Causes, in order of frequency:")
    print("    1. transient credential propagation - Runtime > Restart session, then Run all")
    print("    2. third-party cookies blocked in the browser; Drive auth requires them")
    print("    3. an incognito or embedded window, which cannot complete the auth handshake")
    print("    4. an already-authorised session in another tab holding the credential")
    return False

if not _mount_drive():
    raise RuntimeError("Google Drive could not be mounted - see the diagnosis above. "
                       "All source data lives on Drive, so the run cannot proceed. "
                       "This is an environment fault, not a pipeline fault.")

NEED = ["cik", "year", "ci", "roa", "q", "sga", "size", "lev", "rnd", "capex"]
ALT = {
  "cik": ["cik", "cik_code", "cik_int", "sec_cik"],
  "year": ["fyear", "year", "fiscal_year", "fy", "fiscal_yr", "yr", "report_year"],
  "ci": ["ci_score", "ci", "cc_score", "adherence", "adherence_score", "clean_core_score", "core_integrity", "ccaof", "ccaof_score", "ci_composite", "ci_z", "cc_adherence", "adh_score"],
  "roa": ["roa", "return_on_assets", "roa_w", "ni_at", "return_assets"],
  "q": ["tobins_q", "tobinq", "tobin_q", "tobinsq", "tq", "q_w", "q"],
  "sga": ["sga_sales", "sga", "sga_ratio", "sga_intensity", "sga_rev", "sga_at", "sgna", "xsga_sale"],
  "size": ["log_at", "ln_assets", "log_assets", "lnassets", "at_log", "ln_at", "lnta", "log_ta", "size"],
  "lev": ["leverage", "lev", "debt_ratio", "lev_w", "tl_at", "debt_at", "td_at", "leverage_w"],
  "rnd": ["rnd_int", "rd_intensity", "rnd", "xrd_at", "xrd_sale", "rd_sale", "rnd_at", "rd_at", "r_d"],
  "capex": ["capex_int", "capex", "capx_at", "capx_sale", "capex_at_ratio", "capx", "capex_ratio"],
}
TICK_ALIASES = ["ticker", "tic", "symbol"]
EXTS = (".csv", ".parquet", ".pq", ".feather", ".pkl", ".pickle", ".xlsx", ".dta")

def resolve(cols):
    low = {str(c).lower().strip(): c for c in cols}
    m = {}
    for k, cands in ALT.items():
        hit = next((low[c] for c in cands if c in low), None)
        if hit is not None: m[k] = hit
    return m

def _csv_sep(path):
    """SimFin and many EU exports are ';'-delimited — sniff rather than assume."""
    try:
        with open(path, "r", errors="ignore") as f: head = f.readline()
        return ";" if head.count(";") > head.count(",") else ","
    except Exception:
        return ","

READ_ERRORS = []          # (path, error) - a swallowed exception is a lost diagnosis

def read_head(path, n=400, _tries=3):
    """Drive I/O in Colab fails transiently under load, so retry before giving up - and ALWAYS
    record the reason. Returning a bare None turns one systemic error into 'everything unreadable'."""
    ext = os.path.splitext(path)[1].lower()
    last = None
    for _a in range(_tries):
        try:
            return _read_head_once(path, n, ext)
        except Exception as e:
            last = e
            time.sleep(0.4 * (_a + 1))
    READ_ERRORS.append((path, f"{type(last).__name__}: {str(last)[:120]}"))
    return None, None

def _read_head_once(path, n, ext):
    if True:
        if ext == ".csv":
            df = pd.read_csv(path, nrows=n, low_memory=False, sep=_csv_sep(path)); return list(df.columns), df
        if ext in (".parquet", ".pq"):
            import pyarrow.parquet as pq
            pf = pq.ParquetFile(path)
            cols = [c for c in pf.schema_arrow.names]
            try: smp = pf.read_row_group(0).to_pandas().head(n)
            except Exception: smp = pd.read_parquet(path).head(n)
            return cols, smp
        if ext == ".feather":
            df = pd.read_feather(path); return list(df.columns), df.head(n)
        if ext in (".pkl", ".pickle"):
            if os.path.getsize(path) > 300_000_000: return None, None
            df = pd.read_pickle(path)
            if not isinstance(df, pd.DataFrame): return None, None
            return list(df.columns), df.head(n)
        if ext == ".xlsx":
            df = pd.read_excel(path, nrows=n); return list(df.columns), df
        if ext == ".dta":
            it = pd.read_stata(path, chunksize=n); df = next(it); return list(df.columns), df
    return None, None

def read_full(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv": return pd.read_csv(path, low_memory=False, sep=_csv_sep(path))
    if ext in (".parquet", ".pq"): return pd.read_parquet(path)
    if ext == ".feather": return pd.read_feather(path)
    if ext in (".pkl", ".pickle"): return pd.read_pickle(path)
    if ext == ".xlsx": return pd.read_excel(path)
    if ext == ".dta": return pd.read_stata(path)
    raise ValueError(ext)

def read_cache(path, required=(), label=""):
    """Read a cached CSV only if its SCHEMA still matches what the current code needs.

    A cache written by an earlier version of this notebook can be missing columns that later
    code depends on, which surfaces as an opaque KeyError several cells downstream. Validating
    the schema on read turns that into an automatic, silent recompute - the same self-healing
    contract the rest of the pipeline follows."""
    if not os.path.exists(path):
        return None
    try:
        d_ = pd.read_csv(path)
    except Exception as e:
        print(f"  cache unreadable ({label or os.path.basename(path)}): {type(e).__name__} - recomputing")
        return None
    miss = [c for c in required if c not in d_.columns]
    if miss:
        print(f"  cached {label or os.path.basename(path)} predates current code (missing {miss}) - recomputing")
        return None
    return d_

def read_col(path, col):
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".csv": return pd.read_csv(path, usecols=[col], low_memory=False, sep=_csv_sep(path))[col]
        if ext in (".parquet", ".pq"): return pd.read_parquet(path, columns=[col])[col]
        return read_full(path)[col]
    except Exception:
        try: return read_full(path)[col]
        except Exception: return None

print("Scanning Drive … [discovery v11 · csv/parquet/feather/pkl/xlsx/dta · project-tree-first · content inference · CIK registry validation · derivation layer · prefetch fallback · cascade-proof]")
data_files, cache_dirs, edgar_dirs = [], [], []
for root, dirs, files in os.walk(DRIVE_ROOT):
    dirs[:] = [dd for dd in dirs if not dd.startswith(".")]
    base = os.path.basename(root).lower()
    if base == "eagi_cache": cache_dirs.append(root)
    if PROJECT_HINT in root.lower() and base == "edgar": edgar_dirs.append((root, len(files)))
    if any(x in root.lower() for x in ("eagi_out", "eagi_cache")):
        continue                                    # our own artefacts: never sources, and scanning
    for f in files:                                 # them multiplies Drive I/O for nothing
        if f.lower().endswith(EXTS): data_files.append(os.path.join(root, f))
in_project = [p for p in data_files if PROJECT_HINT in p.lower()]
print(f"  {len(data_files)} data files · {len(in_project)} in '{PROJECT_HINT}' tree · {len(cache_dirs)} eagi_cache dir(s)")
for ed, nf in edgar_dirs:
    print(f"  existing project edgar folder: {ed} ({nf} files) — not auto-wired; share a sample filename to reuse it")

# ---- SEC contact identity: Colab secrets → Drive cache → prompt ----
try:
    from google.colab import userdata
    for _k in ("SEC_EMAIL", "SEC_CONTACT_EMAIL"):
        if not USER_AGENT:
            try:
                _v = userdata.get(_k)
                if _v and "@" in _v: USER_AGENT = _v.strip(); print(f"  SEC identity from Colab secret {_k}")
            except Exception: pass
except Exception: pass
ua_seed = next((os.path.join(cdir, "user_agent.txt") for cdir in cache_dirs if os.path.exists(os.path.join(cdir, "user_agent.txt"))), os.path.join(DRIVE_ROOT, ".eagi_ua.txt"))
if not USER_AGENT and os.path.exists(ua_seed): USER_AGENT = open(ua_seed).read().strip()
while not USER_AGENT or "@" not in USER_AGENT:
    USER_AGENT = input("SEC requires a contact identity. Enter 'Name email@domain': ").strip()
if " " not in USER_AGENT:
    USER_AGENT = "Research Contact " + USER_AGENT   # SEC prefers 'Name email'
open(os.path.join(DRIVE_ROOT, ".eagi_ua.txt"), "w").write(USER_AGENT)

def tick2cik():
    p = os.path.join(DRIVE_ROOT, ".eagi_company_tickers.json")
    if not os.path.exists(p):
        import requests
        r = requests.get("https://www.sec.gov/files/company_tickers.json", headers={"User-Agent": USER_AGENT}, timeout=30)
        r.raise_for_status(); open(p, "w").write(r.text)
    j = json.load(open(p))
    return {v["ticker"].upper(): int(v["cik_str"]) for v in j.values()}

def cik_share(vals):
    """Fraction of candidate identifiers that are real SEC CIKs (registry match)."""
    try:
        u = set(tick2cik().values())
        v = pd.Series(list(vals)).dropna().astype(int)
        return float(v.isin(list(u)).mean()) if len(v) else 0.0
    except Exception:
        return None

def name_score(p):
    n = os.path.basename(p).lower()
    s = 0
    if "panel" in n: s += 3
    if any(k in n for k in ("ccaof", "core_integrity", "coreintegrity", "adher", "clean")): s += 2
    if re.search(r"\bci\b|ci_|_ci", n): s += 1
    if "eagi" in n: s -= 3
    return s

def profile(sample):
    info = {}
    for c in sample.columns:
        try:
            s = sample[c].dropna()
            if len(s) < 20: continue
            if pd.api.types.is_bool_dtype(s): continue
            if pd.api.types.is_numeric_dtype(s):
                s = pd.to_numeric(s, errors="coerce").dropna()
                if len(s) < 20: continue
                info[c] = dict(kind="num", med=float(s.median()), p05=float(s.quantile(.05)), p95=float(s.quantile(.95)),
                               sd=float(s.std() or 0), nun=int(s.nunique()), ints=bool((s % 1 == 0).mean() > .95))
            else:
                u = s.astype(str).str.strip()
                info[c] = dict(kind="obj", nun=int(u.nunique()), tick=bool(u.str.fullmatch(r"[A-Za-z][A-Za-z0-9\.\-]{0,5}").mean() > .8))
        except Exception:
            continue
    return info

def content_guess(info, missing):
    prop, cands, used = {}, {k: [] for k in missing}, set()
    def fits(k, st):
        if st["kind"] != "num": return False
        m, lo, hi = st["med"], st["p05"], st["p95"]
        return {
          "year":  st["ints"] and 1990 <= m <= 2030 and 3 <= st["nun"] <= 45,
          "cik":   st["ints"] and lo >= 1000 and hi <= 2_000_000 and st["nun"] > 50,
          "size":  (not st["ints"]) and 3 <= m <= 15 and 0.4 <= st["sd"] <= 4,
          "lev":   -0.02 <= lo and hi <= 2.5 and 0.02 <= m <= 0.9,
          "roa":   -0.6 <= lo and hi <= 0.6 and -0.15 <= m <= 0.25,
          "q":     lo >= -0.01 and 0.4 <= m <= 6 and hi <= 30,
          "sga":   lo >= -0.01 and hi <= 3 and 0.03 <= m <= 0.8,
          "rnd":   lo >= -0.01 and hi <= 0.8 and m <= 0.2,
          "capex": lo >= -0.01 and hi <= 0.8 and m <= 0.25,
          "ci":    (abs(m) < 0.6 and 0.4 <= st["sd"] <= 2.5) or (lo >= -0.01 and hi <= 1.05) or (lo >= -0.5 and 50 <= hi <= 100.5),
        }.get(k, False)
    KEYTOK = {"cik": {"cik"}, "year": {"year", "fy", "fyear", "fiscal"}, "ci": {"ci", "adher", "ccaof", "clean", "core", "score", "adh"},
              "roa": {"roa", "ni", "margin", "return"}, "q": {"q", "tobin"}, "sga": {"sga", "sgna", "sganda"},
              "size": {"ln", "log", "size", "lnta"}, "lev": {"lev", "leverage", "debt", "liab"},
              "rnd": {"rnd", "rd", "xrd", "research"}, "capex": {"capex", "capx"}}
    def match_name(k, col):
        cl_ = str(col).lower()
        toks = set(re.split(r"[^a-z0-9]+", cl_))
        kt = KEYTOK[k]
        return bool(toks & kt) or any(t in cl_ for t in kt if len(t) >= 3)
    order = [k for k in ["year", "cik", "size", "q", "lev", "sga", "roa", "capex", "rnd", "ci"] if k in missing]
    for k in order:
        cl = [c for c, st in info.items() if c not in used and fits(k, st)]
        cands[k] = cl
        pick = None
        strong = [c for c in cl if match_name(k, c)]
        pool = strong if strong else cl
        if pool:
            sims = [(max((difflib.SequenceMatcher(None, str(c).lower(), a).ratio() for a in ALT[k]), default=0), c) for c in pool]
            sims.sort(reverse=True)
            if len(strong) == 1: pick = strong[0]
            elif strong: pick = sims[0][1]
            elif len(cl) == 1 or sims[0][0] >= 0.72: pick = sims[0][1]
        if pick: prop[k] = pick; used.add(pick)
    tick_cols = [c for c, st in info.items() if st["kind"] == "obj" and st.get("tick") and st["nun"] > 30 and c not in used]
    if "cik" in missing and "cik" not in prop and tick_cols: prop["_ticker"] = tick_cols[0]
    return prop, cands

# ---- Pass 1: project tree (header + content). Pass 2: global (header-only). ----
PANEL_PATH, MAP, TICKER_COL, best = None, {}, None, []
pool1 = sorted(in_project, key=name_score, reverse=True)
pool2 = sorted([p for p in data_files if p not in in_project], key=name_score, reverse=True)[:150]
for phase, pool in ((1, pool1), (2, pool2)):
    for p in pool:
        cols, smp = read_head(p)
        if cols is None: continue
        low = {str(c).lower(): c for c in cols}
        hdr_tick = next((low[a] for a in TICK_ALIASES if a in low), None)
        m = resolve(cols)
        missing = [k for k in NEED if k not in m]
        cg = {}
        if phase == 1 and missing and smp is not None:
            cg, _c = content_guess(profile(smp), missing)
            for k, c in cg.items():
                if k != "_ticker": m[k] = c
            missing = [k for k in NEED if k not in m]
            if missing == ["cik"] and "_ticker" in cg: TICKER_COL = cg["_ticker"]; missing = []
        best.append((len(NEED) - len(missing), p, missing, m, cg.get("_ticker") or hdr_tick))
        if missing and all(k in ("capex",) for k in missing):
            print(f"  accepting {os.path.basename(p)} - only optional field(s) {missing} absent")
            missing = []
        if not missing:
            PANEL_PATH, MAP = p, m
            break
    if PANEL_PATH: break

# ---- CI-split rescue (content-based, overlap-validated) ----
CI_PATH, CI_MAP, CI_TICK = (CI_FILE_OVERRIDE or None), {}, None
def find_ci_file(exclude, anchor_ciks=None):
    """Any project file with year + (cik|ticker) + a CI-shaped column; validated by firm overlap."""
    for p2 in pool1:
        if p2 in exclude: continue
        c2, s2 = read_head(p2)
        if c2 is None: continue
        m2 = resolve(c2)
        low2 = {str(c).lower(): c for c in c2}
        t2 = next((low2[a] for a in TICK_ALIASES if a in low2), None)
        if s2 is not None and ("ci" not in m2 or "year" not in m2):
            gg, _cc = content_guess(profile(s2), [k for k in ("ci", "year", "cik") if k not in m2])
            for k in ("ci", "year", "cik"):
                if k in gg: m2[k] = gg[k]
            t2 = t2 or gg.get("_ticker")
        if not ("ci" in m2 and "year" in m2 and ("cik" in m2 or t2)):
            continue
        if anchor_ciks:
            try:
                if "cik" in m2:
                    ids = pd.to_numeric(read_col(p2, m2["cik"]), errors="coerce").dropna().astype(int)
                else:
                    ids = read_col(p2, t2).astype(str).str.upper().map(tick2cik()).dropna().astype(int)
                if ids.isin(list(anchor_ciks)).mean() < 0.5:
                    continue
            except Exception:
                continue
        return p2, m2, t2
    return None, {}, None

if PANEL_PATH is None:
    best.sort(key=lambda t: t[0], reverse=True)
    for score, p, missing, m, tk in best[:12]:
        if set(missing) <= {"ci"} or (set(missing) <= {"ci", "cik"} and tk):
            anchor = None
            try:
                if "cik" in m:
                    anchor = set(pd.to_numeric(read_col(p, m["cik"]), errors="coerce").dropna().astype(int).unique())
            except Exception:
                anchor = None
            p2, m2, t2 = find_ci_file({p}, anchor)
            if p2:
                PANEL_PATH, MAP, TICKER_COL = p, m, tk
                CI_PATH, CI_MAP, CI_TICK = p2, m2, t2
                print(f"CI-split rescue: fundamentals {p} + CI {p2}")
                break

# ---- CIK sanity: reject identifier columns that are not SEC CIKs ----
CIK_SHARE = None
if PANEL_PATH is not None and "cik" in MAP:
    _v = read_col(PANEL_PATH, MAP["cik"])
    if _v is not None:
        CIK_SHARE = cik_share(pd.to_numeric(_v, errors="coerce").dropna().astype(int).unique())
        if CIK_SHARE is not None:
            print(f"CIK column '{MAP['cik']}' → SEC registry match {CIK_SHARE:.0%}")
            if CIK_SHARE < 0.5:
                cols_p, _sx = read_head(PANEL_PATH, 5)
                lowp = {str(c).lower(): c for c in (cols_p or [])}
                tcol = next((lowp[a] for a in TICK_ALIASES if a in lowp), None)
                if tcol:
                    print(f"→ rejected as CIK; using ticker column '{tcol}' via the SEC map instead")
                    MAP.pop("cik", None); TICKER_COL = tcol
                else:
                    print("⚠ identifier is likely NOT a CIK and no ticker column exists — expect mass skips; paste the RUN DIAGNOSTIC block")

# ============ DERIVATION LAYER ============
# The source panel stores CONSTITUENTS, not canonical fields: Core Integrity lives in the five
# CIAOF KPI columns, and outcomes/controls are raw line items. Derive rather than hunt.
KPI_NAMES = ["standardization_kpi", "data_harmonization_kpi", "api_integration_kpi",
             "modular_extensibility_kpi", "ops_automation_kpi"]
OUTCOMES  = ["roa", "q", "sga"]             # dependent variables: required per-regression, NOT panel-wide
NEED_OPT  = ["capex"]                       # nice-to-have control: absence must not block the run
NEED_CORE = [k for k in NEED if k not in NEED_OPT + OUTCOMES]   # cik, year, ci, size, lev, rnd
FIELD_PATS = {
  "assets":    [r"^assets$", r"^assets[_ ]?sf$", r"total[_ ]?assets", r"^at$"],
  "revenue":   [r"^revenue$", r"^revenue[_ ]?sf$", r"^sales$", r"total[_ ]?revenue", r"^revt$"],
  "netinc":    [r"^net[_ ]?income$", r"^net[_ ]?income[_ ]?sf$", r"net[_ ]?income", r"^ni$"],
  "liab":      [r"total[_ ]?liabilities$", r"^liabilities$", r"^liab", r"^lt$", r"total[_ ]?liab"],
  "sga_exp":   [r"sga[_ ]?expense", r"^sga$", r"^sga[_ ]?sf$", r"selling.*general"],
  "rd_exp":    [r"rd[_ ]?expense", r"^rd[_ ]?sf$", r"research.*development", r"^xrd$"],
  "capex_exp": [r"^capex", r"capital[_ ]?expenditure", r"^capx$", r"purchase.*fixed[_ ]?assets"],
  "mktcap":    [r"^mktcap", r"market[_ ]?cap", r"market[_ ]?value.*equity"],
  "tobin":     [r"tobin"],
  "ci_col":    [r"^ccai[_ ]?equal$", r"^ccai$", r"^ci[_ ]?score$", r"^core[_ ]?integrity$"],
  "ci_alt":    [r"^ccai[_ ]?pca$", r"^ccai[_ ]?textrich$"],
  "roa_col":   [r"^roa$", r"return[_ ]?on[_ ]?assets"],
  "lev_col":   [r"^leverage$", r"debt[_ ]?ratio", r"^lev$"],
  "sga_col":   [r"^sga[_ ]?intensity", r"^sga[_ ]?ratio"],
  "rnd_col":   [r"^rd[_ ]?intensity", r"^rnd[_ ]?int", r"^rd[_ ]?sale"],
  "size_col":  [r"^log[_ ]?assets$", r"^ln[_ ]?assets$", r"log[_ ]?at$"],
  "capsoft":   [r"^capsoft[_ ]?intensity", r"capitali[sz]ed[_ ]?software"],
  "cik_col":   [r"^cik$", r"^cik[_ ]?code$", r"^sec[_ ]?cik$"],
  "tick_col":  [r"^ticker$", r"^tic$", r"^symbol$"],
}

AUX_PATS = {   # adjacent in-panel signals: convergent-validity checks against EAGI
  "aux_gov_it":        [r"^gov[_ ]?it[_ ]?signal", r"gov.*it.*signal"],
  "aux_it_intensity":  [r"^it[_ ]?intensity$"],
  "aux_ccai_textrich": [r"^ccai[_ ]?textrich$"],
  "aux_n_tokens":      [r"^n[_ ]?tokens$"],
  "aux_gh_api":        [r"^gh[_ ]?api[_ ]?ratio$"],
  "aux_gh_modular":    [r"^gh[_ ]?modular[_ ]?ratio$"],
  "aux_gh_ops":        [r"^gh[_ ]?ops[_ ]?ratio$"],
}

def _pick(cols, pats, prefer_sf=True):
    """First pattern match; among matches prefer the SimFin (_sf) family so ratio
    numerators and denominators come from the same reporting source."""
    low = [(str(c).lower(), c) for c in cols]
    hits = []
    for pat in pats:
        for lc, orig in low:
            if re.search(pat, lc) and orig not in hits: hits.append(orig)
    if not hits: return None
    if prefer_sf:
        sf = [h for h in hits if str(h).lower().endswith("_sf")]
        if sf: return sf[0]
    return hits[0]

def _num(fr, c):
    return pd.to_numeric(fr[c], errors="coerce")

def cronbach(X):
    X = X.dropna()
    k = X.shape[1]
    if k < 2 or len(X) < 10: return None
    tot = X.sum(axis=1).var(ddof=1)
    if not tot: return None
    return float(k / (k - 1) * (1 - X.var(ddof=1).sum() / tot))

def derive_fields(fr):
    """Return ({canonical: Series}, {canonical: provenance string})."""
    got, how = {}, {}
    C = list(fr.columns)
    f = {k: _pick(C, p) for k, p in FIELD_PATS.items()}
    # --- Core Integrity: prefer the PUBLISHED composite over any recomputation ---
    kpi = [c for c in C if str(c).lower() in KPI_NAMES] or [c for c in C if str(c).lower().endswith("_kpi")]
    alpha = None
    if len(kpi) >= 2:
        alpha = cronbach(pd.concat([_num(fr, c) for c in kpi], axis=1))
    if f["ci_col"]:
        got["ci"] = _num(fr, f["ci_col"])
        how["ci"] = f"published composite '{f['ci_col']}'" + (f" · KPI α={alpha:.3f}" if alpha is not None else "")
    elif len(kpi) >= 3:
        X = pd.concat([_num(fr, c) for c in kpi], axis=1); X.columns = [str(c) for c in kpi]
        got["ci"] = ((X - X.mean()) / X.std().replace(0, 1)).mean(axis=1)
        how["ci"] = f"z-mean of {len(kpi)} CIAOF KPIs" + (f" · α={alpha:.3f}" if alpha is not None else "")
    if f["ci_alt"]:
        got["ci_alt"] = _num(fr, f["ci_alt"]); how["ci_alt"] = f"robustness composite '{f['ci_alt']}'"
    P = lambda pats: _pick(C, pats, prefer_sf=False)
    c = dict(
        assets_sf=P([r"^assets[_ ]?sf$"]), assets=P([r"^assets$"]),
        rev_sf=P([r"^revenue[_ ]?sf$"]),   rev=P([r"^revenue$"]),
        ni_sf=P([r"^net[_ ]?income[_ ]?sf$"]), ni=P([r"^net[_ ]?income$"]),
        liab_sf=P([r"^liab[_ ]?sf$"]), liab=P([r"total[_ ]?liabilities$", r"^liabilities$"]),
        sga_int=P([r"^sga[_ ]?intensity$"]), sga_sf=P([r"^sga[_ ]?sf$"]), sga_exp=P([r"^sga[_ ]?expense$"]),
        rd_int=P([r"^rd[_ ]?intensity[_ ]?sf$", r"^rd[_ ]?intensity$"]), rd_sf=P([r"^rd[_ ]?sf$"]), rd_exp=P([r"^rd[_ ]?expense$"]),
        mkt_sf=P([r"^mktcap[_ ]?sf$"]), tobin=P([r"tobin"]),
        nm_sf=P([r"^net[_ ]?margin[_ ]?sf$"]), at_sf=P([r"^asset[_ ]?turnover[_ ]?sf$"]),
        logat=P([r"^log[_ ]?assets$", r"^ln[_ ]?assets$"]),
    )
    def cc(k):
        return _num(fr, c[k]) if c.get(k) else None
    def rat(n_, d_, absolute=False):
        if n_ is None or d_ is None: return None
        return (n_.abs() if absolute else n_) / d_.replace(0, np.nan)
    def coal(name, *cands):
        """Fill gaps across ordered fallbacks, recording HOW MANY rows each source contributed
        so measurement heterogeneity is auditable rather than hidden."""
        ser, used = None, []
        for lbl, s_ in cands:
            if s_ is None: continue
            if ser is None:
                ser = s_.copy(); used.append((lbl, int(ser.notna().sum())))
            else:
                before = int(ser.notna().sum())
                ser = ser.fillna(s_)
                used.append((lbl, int(ser.notna().sum()) - before))
        if ser is not None:
            got[name] = ser
            live = [(l, k) for l, k in used if k > 0]
            how[name] = " + ".join(f"{l} [{k} rows]" for l, k in live) + (" (coalesced)" if len(live) > 1 else "")
        return ser

    A = cc("assets_sf") if c.get("assets_sf") else cc("assets")
    A2 = cc("assets") if c.get("assets") else None
    R = cc("rev_sf") if c.get("rev_sf") else cc("rev")
    R2 = cc("rev") if c.get("rev") else None

    coal("size", (f"direct '{c['logat']}'", cc("logat")),
                 (f"log({c.get('assets_sf') or c.get('assets')})", np.log(A.clip(lower=1)) if A is not None else None),
                 (f"log({c.get('assets')})", np.log(A2.clip(lower=1)) if A2 is not None else None))
    coal("roa", (f"direct '{f['roa_col']}'", cc("roa_col") if f["roa_col"] else None),
                (f"{c['ni_sf']}/{c.get('assets_sf')}", rat(cc("ni_sf"), A)),
                (f"{c['nm_sf']}×{c['at_sf']}", (cc("nm_sf") * cc("at_sf")) if (c.get("nm_sf") and c.get("at_sf")) else None),
                (f"{c['ni']}/{c.get('assets')}", rat(cc("ni"), A2)))
    coal("lev", (f"direct '{f['lev_col']}'", cc("lev_col") if f["lev_col"] else None),
                (f"{c['liab_sf']}/{c.get('assets_sf')}", rat(cc("liab_sf"), A)),
                (f"{c['liab']}/{c.get('assets')}", rat(cc("liab"), A2)))
    coal("sga", (f"direct '{c['sga_int']}'", cc("sga_int")),
                (f"|{c['sga_sf']}|/{c.get('rev_sf')}", rat(cc("sga_sf"), R, absolute=True)),
                (f"|{c['sga_exp']}|/{c.get('rev')}", rat(cc("sga_exp"), R2, absolute=True)))
    coal("rnd", (f"direct '{c['rd_int']}'", cc("rd_int")),
                (f"|{c['rd_sf']}|/{c.get('rev_sf')}", rat(cc("rd_sf"), R, absolute=True)),
                (f"|{c['rd_exp']}|/{c.get('rev')}", rat(cc("rd_exp"), R2, absolute=True)))
    coal("q", (f"direct '{c['tobin']}'", cc("tobin")),
              (f"({c['mkt_sf']}+{c['liab_sf']})/{c.get('assets_sf')}",
               rat(cc("mkt_sf") + (cc("liab_sf") if c.get("liab_sf") else 0), A) if c.get("mkt_sf") else None))
    # R&D: absence of an R&D line is a reported zero, not missing data (standard convention)
    if "rnd" in got:
        _m = got["rnd"].isna()
        got["rnd_missing"] = _m.astype(float)
        got["rnd"] = got["rnd"].fillna(0.0)
        how["rnd"] += f" · missing→0 for {int(_m.sum())} rows + rnd_missing dummy"
        how["rnd_missing"] = "indicator: firm-year reports no R&D line"
    if f["capsoft"]:
        got["capsoft"] = _num(fr, f["capsoft"]); how["capsoft"] = f"capital-intensity proxy '{f['capsoft']}' (cash capex unavailable)"
    if f["capex_exp"] and A is not None:
        got["capex"] = _num(fr, f["capex_exp"]).abs() / A.replace(0, np.nan); how["capex"] = f"|{f['capex_exp']}|/{c.get('assets_sf')}"
    for _kc in [c for c in C if str(c).lower().endswith("_kpi")]:
        got["kpi_" + str(_kc)] = _num(fr, _kc)
        how["kpi_" + str(_kc)] = f"CIAOF dimension '{_kc}' (carried for dimension-level tests)"
    _ind = _pick(C, [r"^industry$", r"^sector$", r"^sic", r"^gics"], prefer_sf=False)
    if _ind is not None:
        got["industry_raw"] = fr[_ind].astype(str); how["industry_raw"] = f"industry label '{_ind}'"
    for _ak, _ap in AUX_PATS.items():
        _c = _pick(C, _ap, prefer_sf=False)
        if _c is not None:
            got[_ak] = _num(fr, _c); how[_ak] = f"validation signal '{_c}'"
    return got, how

def simfin_id_to_cik():
    """Bridge SimFinId → Ticker → CIK using the SimFin cache (semicolon-delimited)."""
    for p in pool1:
        if not p.lower().endswith(".csv") or _is_ours(p): continue
        try:
            fr = pd.read_csv(p, sep=_csv_sep(p), nrows=300000, low_memory=False)
        except Exception:
            continue
        low = {str(c).lower().replace(" ", ""): c for c in fr.columns}
        if "simfinid" in low and "ticker" in low:
            t2c = tick2cik()
            mm = fr[[low["simfinid"], low["ticker"]]].drop_duplicates()
            mm["_cik"] = mm[low["ticker"]].astype(str).str.upper().map(t2c)
            mm = mm.dropna(subset=["_cik"])
            print(f"  SimFin bridge from {os.path.basename(p)}: {len(mm)} ids → CIK")
            return {str(k): int(v) for k, v in zip(mm[low["simfinid"]], mm["_cik"])}
    return {}

CACHE_ARTIFACTS = ("term_counts.csv", "techcom_raw.csv", "panel_with_eagi.csv",
                   "eagi_main_results.csv", "eagi_loto.csv", "company_tickers.json")

def _is_ours(p):
    """Never treat our own caches/outputs as source data."""
    pl = p.lower()
    return ("eagi_cache" in pl) or ("eagi_out" in pl) or (os.path.basename(pl) in CACHE_ARTIFACTS)

def find_cik_series(fr):
    """Locate CIKs by CONTENT (SEC registry match), never by column name."""
    try:
        tkmap = tick2cik(); reg = set(tkmap.values())
    except Exception as e:
        print("  registry unavailable:", type(e).__name__); return None, None
    bestc, bests = None, 0.0
    for c in fr.columns:
        try:
            v = pd.to_numeric(fr[c], errors="coerce").dropna()
            if len(v) < 10: continue
            if (v % 1 == 0).mean() < 0.99: continue
            if v.min() < 1 or v.max() > 2_000_000: continue
            sh = float(v.astype(int).isin(reg).mean())
            if sh > bests: bestc, bests = c, sh
        except Exception:
            continue
    if bestc is not None and bests >= 0.9:
        print(f"  CIK by content → column '{bestc}' · registry match {bests:.0%}")
        return pd.to_numeric(fr[bestc], errors="coerce"), f"content:{bestc}"
    for c in fr.columns:
        try:
            u = fr[c].astype(str).str.upper().str.strip()
            sh = float(u.isin(tkmap.keys()).mean())
            if sh >= 0.8:
                print(f"  CIK via ticker column '{c}' · {sh:.0%} of rows map")
                return u.map(tkmap), f"ticker:{c}"
        except Exception:
            continue
    if bestc is not None:
        print(f"  best numeric identifier '{bestc}' only {bests:.0%} registry match — rejected")
    return None, None

def assemble_panel():
    """Merge project tables on (firm key, year), derive canonical fields, attach real CIKs."""
    frames = []
    for p in pool1:
        if _is_ours(p): continue
        cols, _s = read_head(p, 5)
        if cols is None: continue
        low = {str(c).lower(): c for c in cols}
        kid = next((low[k] for k in ("firm_id", "cik", "ticker", "tic", "symbol") if k in low), None)
        yid = next((low[k] for k in ("year", "fyear", "fiscal_year", "fy", "fiscal year") if k in low), None)
        if not (kid and yid): continue
        try:
            fr = read_full(p)
        except Exception as e:
            print("  skip", os.path.basename(p), type(e).__name__); continue
        if len(fr) > 400_000:
            print("  skip (too large)", os.path.basename(p)); continue
        fr = fr.copy()
        fr["_fid"] = fr[kid].astype(str)
        fr["_yr"] = pd.to_numeric(fr[yid], errors="coerce")
        fr = fr.dropna(subset=["_yr"])
        if not len(fr): continue
        fr["_yr"] = fr["_yr"].astype(int)
        fr = fr.drop_duplicates(subset=["_fid", "_yr"])
        frames.append((p, fr))
    if not frames:
        print("  no table carries a firm key + year — derivation impossible"); return None
    frames.sort(key=lambda t: -t[1].shape[1])
    base_path, merged = frames[0][0], frames[0][1].copy()
    print(f"  base: {os.path.basename(base_path)} — {merged.shape[0]} rows × {merged.shape[1]} cols")
    print(f"  base columns: {[c for c in merged.columns if not str(c).startswith('_')]}")
    keys = set(merged["_fid"])
    for p, fr in frames[1:]:
        ov = float(fr["_fid"].isin(keys).mean())
        if ov < 0.2:
            print(f"    – {os.path.basename(p)}: key overlap {ov:.0%} → skipped"); continue
        newc = [c for c in fr.columns if c not in merged.columns]
        if not newc: continue
        try:
            merged = merged.merge(fr[["_fid", "_yr"] + newc], on=["_fid", "_yr"], how="left")
            print(f"    + {os.path.basename(p)} (overlap {ov:.0%}, +{len(newc)} cols)")
        except Exception:
            continue
    got, how = derive_fields(merged)
    globals()["DERIV_HOW"] = how
    out = pd.DataFrame({"year": merged["_yr"].values})
    ciks, prov = find_cik_series(merged)
    if ciks is None:
        bridge = simfin_id_to_cik()
        if bridge:
            ciks = merged["_fid"].astype(str).map(bridge)
            print(f"  SimFinId bridge → CIK for {ciks.notna().mean():.0%} of rows")
            if ciks.notna().mean() < 0.5: ciks = None
    if ciks is None:
        print("  ✗ no usable CIK — cannot align the panel to the filing text layer")
        print("   full merged column list:", list(merged.columns))
        return None
    out["cik"] = ciks.values
    for k, v in got.items(): out[k] = v.values if hasattr(v, "values") else v
    print("  derived fields:")
    for k in NEED + ["capsoft", "ci_alt"] + [a for a in AUX_PATS if a in out.columns]:
        print(f"    {k:6s} ← " + (how.get(k, "(direct)") if k in out.columns else "MISSING"))
    out = out.dropna(subset=["cik", "year"])
    out["cik"] = out["cik"].astype(int); out["year"] = out["year"].astype(int)
    _dup = int(out.duplicated(subset=["cik", "year"]).sum())
    if _dup:
        print(f"  ⚠ {_dup} duplicate (cik,year) rows — collapsing by mean "
              f"(distinct firm_ids sharing one CIK, e.g. dual share classes)")
        _ind_keep = (out.groupby(["cik", "year"], as_index=False)["industry_raw"].first()
                     if "industry_raw" in out.columns else None)
        out = out.groupby(["cik", "year"], as_index=False).mean(numeric_only=True)
        if _ind_keep is not None:
            out = out.merge(_ind_keep, on=["cik", "year"], how="left")
    miss = [k for k in NEED_CORE if k not in out.columns]
    for k in NEED_OPT:
        if k not in out.columns: print(f"  note: optional control '{k}' unavailable — dropped from the control set")
    if miss:
        print(f"  ✗ still missing {miss} — full merged column list:")
        print("   ", list(merged.columns))
        return None
    print(f"  field coverage before complete-case filter ({len(out)} rows):")
    for _k in NEED_CORE + OUTCOMES + [x for x in ("capex", "capsoft", "rnd_missing") if x in out.columns]:
        print(f"    {_k:8s} {int(out[_k].notna().sum()):5d}  ({out[_k].notna().mean():.0%})")
    _full = out.dropna(subset=NEED_CORE)
    for _k in NEED_CORE:
        _alt = out.dropna(subset=[c for c in NEED_CORE if c != _k])
        if len(_alt) > 1.2 * max(1, len(_full)):
            print(f"    ⚠ excluding '{_k}' would raise the panel {len(_full)} → {len(_alt)} rows")
    if "industry_raw" in out.columns: out["industry_raw"] = out["industry_raw"].fillna("unknown")
    out = out.dropna(subset=NEED_CORE)
    _oc = [o for o in OUTCOMES if o in out.columns]
    if _oc:
        out = out[out[_oc].notna().any(axis=1)]
        print("  outcome availability inside the panel: " + " · ".join(f"{o}={int(out[o].notna().sum())}" for o in _oc))
    print(f"  ✓ DERIVED PANEL: {len(out)} firm-years · {out.cik.nunique()} firms · {out.year.min()}–{out.year.max()}")
    return out

DERIVED_PANEL, DERIVE_LOG = None, ""
if PANEL_PATH is None:
    import io, contextlib
    print("\nDirect resolution failed → DERIVING canonical panel from project tables …")
    _buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(_buf):
            DERIVED_PANEL = assemble_panel()
    except Exception as e:
        _buf.write(f"  derivation error: {type(e).__name__}: {e}\n")
    DERIVE_LOG = _buf.getvalue()
    print(DERIVE_LOG)

# ---- Prefetch-only fallback ----
PREFETCH_ONLY, FIRM_CIKS = False, None
if PANEL_PATH is None and DERIVED_PANEL is None:
    print("\n──────── PROJECT DATA INVENTORY (full column lists) ────────")
    for p in pool1[:90]:
        cols, _ = read_head(p, n=5)
        rel = p.replace(DRIVE_ROOT + "/", "")
        print(f"· {rel}\n    cols[{0 if cols is None else len(cols)}]: {list(cols) if cols else 'UNREADABLE'}")
    if READ_ERRORS:
        from collections import Counter
        kinds = Counter(e.split(":")[0] for _, e in READ_ERRORS)
        print(f"\n  ⚠ {len(READ_ERRORS)} files could not be read. Error types: {dict(kinds)}")
        for _pp, _ee in READ_ERRORS[:5]:
            print(f"      {os.path.basename(_pp)}: {_ee}")
        print("  If EVERY file failed, this is an ENVIRONMENT fault, not a data fault. The usual")
        print("  cause is pip upgrading numpy/pandas inside a live kernel: Runtime → Restart")
        print("  session, then Run all. If it recurs after a restart, the error types above")
        print("  identify the real cause - do not assume pip.")
    best.sort(key=lambda t: t[0], reverse=True)
    if best:
        s0 = best[0]
        print(f"\nBest candidate: {s0[1]} — resolved {s0[0]}/{len(NEED)}, missing {s0[2]}")
    for score, p, missing, m, tk in best:
        ciks = None
        if "cik" in m:
            v = read_col(p, m["cik"])
            if v is not None: ciks = pd.to_numeric(v, errors="coerce").dropna().astype(int)
        if ciks is None and tk:
            v = read_col(p, tk)
            if v is not None:
                try:
                    t2c = tick2cik()
                    ciks = v.astype(str).str.upper().map(t2c).dropna().astype(int)
                except Exception as e:
                    print("ticker→CIK unavailable:", type(e).__name__)
        if ciks is not None:
            _sh = cik_share(ciks.unique())
            if _sh is not None:
                if _sh < 0.5:
                    print(f"  {p}: identifier rejected as CIK ({_sh:.0%} SEC-registry match)")
                    ciks = None
                elif _sh < 0.9:
                    u = set(tick2cik().values()); n0 = ciks.nunique()
                    ciks = ciks[ciks.isin(list(u))]
                    print(f"  {p}: filtered to SEC-valid CIKs {ciks.nunique()}/{n0} (registry match {_sh:.0%})")
                CIK_SHARE = _sh
        if ciks is not None and ciks.nunique() >= 20:
            FIRM_CIKS = sorted(ciks.unique().tolist())
            print(f"\nPLAN B: firm universe from {p} → {len(FIRM_CIKS)} CIKs. Text-layer prefetch will run and cache to Drive.")
            PREFETCH_ONLY = True
            DATA_DIR = os.path.dirname(p)
            break
    if FIRM_CIKS is None:
        raise RuntimeError("Panel unresolved AND no CIK/ticker column found anywhere — paste the inventory above; mapping is one COLMAP line.")
elif DERIVED_PANEL is not None:
    FIRM_CIKS = sorted(DERIVED_PANEL.cik.unique().tolist())
    CIK_SHARE = cik_share(FIRM_CIKS)
    DATA_DIR = os.path.join(DRIVE_ROOT, PROJECT_HINT + "-research", "data", "processed")
    if not os.path.isdir(DATA_DIR):
        DATA_DIR = os.path.dirname(pool1[0]) if pool1 else DRIVE_ROOT
else:
    DATA_DIR = os.path.dirname(PANEL_PATH)

CACHE_DIR = max(cache_dirs, key=lambda dd: len(os.listdir(dd)), default=os.path.join(DATA_DIR, "eagi_cache"))
OUT_DIR = os.path.join(DATA_DIR, "eagi_out")
os.makedirs(CACHE_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
open(os.path.join(CACHE_DIR, "user_agent.txt"), "w").write(USER_AGENT)

if PREFETCH_ONLY:
    print(f"\nMODE → PREFETCH-ONLY · caches → {CACHE_DIR}")
elif DERIVED_PANEL is not None:
    print(f"\nMODE → DERIVED PANEL · {len(DERIVED_PANEL)} firm-years · {len(FIRM_CIKS)} firms · registry match {CIK_SHARE:.0%}")
    print(f"CACHE  → {CACHE_DIR}  ({len(os.listdir(CACHE_DIR))} files reused)")
else:
    print(f"\nPANEL  → {PANEL_PATH}")
    print(f"MAP    → {MAP}" + (f"  · ticker column: {TICKER_COL}" if TICKER_COL else ""))
    if CI_PATH: print(f"CI     → {CI_PATH}  map {CI_MAP}" + (f" · ticker {CI_TICK}" if CI_TICK else ""))
    print(f"CACHE  → {CACHE_DIR}  ({len(os.listdir(CACHE_DIR))} files reused)")


In [ ]:
# ============ 2 · LOAD PANEL (skipped in prefetch-only mode) ============
def load_panel():
    df = read_full(PANEL_PATH)
    if COLMAP: df = df.rename(columns={v: k for k, v in COLMAP.items()})
    df = df.rename(columns={v: k for k, v in MAP.items()})
    if "cik" not in df.columns and TICKER_COL:
        t2c = tick2cik()
        df["cik"] = df[TICKER_COL].astype(str).str.upper().map(t2c)
        n0 = len(df); df = df.dropna(subset=["cik"])
        print(f"ticker→CIK: {len(df)}/{n0} rows mapped")
    if CI_PATH:
        ci = read_full(CI_PATH).rename(columns={v: k for k, v in CI_MAP.items()})
        if "cik" not in ci.columns and CI_TICK:
            t2c = tick2cik(); ci["cik"] = ci[CI_TICK].astype(str).str.upper().map(t2c); ci = ci.dropna(subset=["cik"])
        ci["cik"] = pd.to_numeric(ci["cik"], errors="coerce"); ci["year"] = pd.to_numeric(ci["year"], errors="coerce")
        ci = ci.dropna(subset=["cik", "year"]); ci["cik"] = ci["cik"].astype(int); ci["year"] = ci["year"].astype(int)
        df = df.merge(ci[["cik", "year", "ci"]], on=["cik", "year"], how="inner")
        print(f"CI merged from separate file → {len(df)} firm-years")
    missing = [c for c in NEED if c not in df.columns]
    assert not missing, f"Panel missing {missing} after mapping — set COLMAP."
    df["cik"] = pd.to_numeric(df["cik"], errors="coerce"); df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df = df.dropna(subset=["cik", "year"])
    df["cik"] = df["cik"].astype(int); df["year"] = df["year"].astype(int)
    out = df[(df.year >= YEAR_MIN) & (df.year <= YEAR_MAX)].copy()
    assert out.cik.nunique() >= 30 and out.year.nunique() >= 3, (
        f"Resolved table looks wrong: {out.cik.nunique()} firms × {out.year.nunique()} years from {PANEL_PATH} — "
        "likely mis-detected; set COLMAP / CI_FILE_OVERRIDE and paste the RUN DIAGNOSTIC block.")
    return out

if G("PREFETCH_ONLY") is None:
    panel = None
    print("UPSTREAM INCOMPLETE — cell 1 did not finish. See the RUN DIAGNOSTIC cell at the end.")
elif G("DERIVED_PANEL") is not None:
    panel = DERIVED_PANEL[(DERIVED_PANEL.year >= YEAR_MIN) & (DERIVED_PANEL.year <= YEAR_MAX)].copy()
    FIRM_CIKS = sorted(panel.cik.unique().tolist())
    print(panel.shape, "firm-years ·", panel.cik.nunique(), "firms  [derived]")
elif PREFETCH_ONLY:
    panel = None
    print(f"PREFETCH-ONLY — {len(FIRM_CIKS)} firms; estimation cells will self-skip until the panel is mapped.")
else:
    panel = load_panel()
    FIRM_CIKS = sorted(panel.cik.unique().tolist())
    _sh = cik_share(FIRM_CIKS)
    if _sh is not None:
        globals()["CIK_SHARE"] = _sh
        if _sh < 0.9:
            _u = set(tick2cik().values()); _n0 = len(FIRM_CIKS)
            FIRM_CIKS = [c for c in FIRM_CIKS if c in _u]
            print(f"CIK sanity: kept {len(FIRM_CIKS)}/{_n0} SEC-valid CIKs (registry match {_sh:.0%})")
        else:
            print(f"CIK sanity: registry match {_sh:.0%}")
    print(panel.shape, "firm-years ·", panel.cik.nunique(), "firms")

In [ ]:
# ============ 3 · EDGAR FETCHERS (cached, rate-limited, lazy headers) ============
import requests
from bs4 import BeautifulSoup

def _hd():
    return {"User-Agent": G("USER_AGENT") or "", "Accept-Encoding": "gzip, deflate"}

def _get(url, binary=False):
    time.sleep(SLEEP)
    r = requests.get(url, headers=_hd(), timeout=30)
    r.raise_for_status()
    return r.content if binary else r.text

def submissions(cik):
    p = f"{CACHE_DIR}/sub_{cik}.json"
    if os.path.exists(p):
        return json.load(open(p))
    j = json.loads(_get(f"https://data.sec.gov/submissions/CIK{int(cik):010d}.json"))
    json.dump(j, open(p, "w"))
    return j

def filings_of(cik, form):
    j = submissions(cik); rec = j["filings"]["recent"]
    for form_i, acc, doc, fdate, rdate in zip(rec["form"], rec["accessionNumber"], rec["primaryDocument"], rec["filingDate"], rec.get("reportDate", rec["filingDate"])):
        if form_i != form:
            continue
        ry = (rdate or fdate)[:4]
        yield int(ry), acc.replace("-", ""), doc

def doc_text(cik, acc, doc):
    p = f"{CACHE_DIR}/txt_{cik}_{acc}.txt"
    if os.path.exists(p):
        return open(p, encoding="utf-8", errors="ignore").read()
    url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc}/{doc}"
    html = _get(url)
    text = BeautifulSoup(html, "lxml").get_text(" ").lower()
    text = re.sub(r"\s+", " ", text)
    open(p, "w", encoding="utf-8").write(text)
    return text

print("fetchers ready — raw filings cached under", G("CACHE_DIR"))

In [ ]:
# ============ 5f · FILING-METADATA INSTRUMENTS — free, mandatory, high base rate ============
# Everything here is already inside the cached submissions JSON. No new documents, no parsing of
# prose, no discretion. The strongest item is FILING LAG: days from period end to filing.
# Closing the books fast is downstream of system quality — it is continuous, universal, and
# recorded for every firm every year, which is exactly what the text index was not.
META_NOTE, META_TBL = [], None
if G("FIRM_CIKS") is None or G("CACHE_DIR") is None:
    print("SKIPPED — no cached submissions.")
else:
    rows, prof = [], []
    for ck in FIRM_CIKS:
        fp = f"{CACHE_DIR}/sub_{ck}.json"
        if not os.path.exists(fp): continue
        try:
            j = json.load(open(fp))
        except Exception:
            continue
        prof.append({"cik": ck,
                     "sic": str(j.get("sic") or ""), "sic_desc": str(j.get("sicDescription") or ""),
                     "fye": str(j.get("fiscalYearEnd") or ""),
                     "filer_category": str(j.get("category") or ""),
                     "state_inc": str(j.get("stateOfIncorporation") or ""),
                     "n_former_names": len(j.get("formerNames") or [])})
        rec = j.get("filings", {}).get("recent", {})
        forms = rec.get("form", []); fdt = rec.get("filingDate", []); rdt = rec.get("reportDate", fdt)
        for f_, d_, r_ in zip(forms, fdt, rdt):
            if not d_: continue
            yy = int(d_[:4])
            if not (YEAR_MIN - 1 <= yy <= YEAR_MAX): continue
            base = {"cik": ck, "form": f_, "fdate": d_, "rdate": r_}
            rows.append(base)
    if rows:
        R = pd.DataFrame(rows)
        R["fdate"] = pd.to_datetime(R["fdate"], errors="coerce")
        R["rdate"] = pd.to_datetime(R["rdate"], errors="coerce")
        ann = R[R["form"] == "10-K"].dropna(subset=["fdate", "rdate"]).copy()
        ann["year"] = ann["rdate"].dt.year
        ann["filing_lag"] = (ann["fdate"] - ann["rdate"]).dt.days
        ann = ann[(ann["filing_lag"] >= 0) & (ann["filing_lag"] <= 400)]
        LAG = ann.groupby(["cik", "year"], as_index=False)["filing_lag"].min()
        late = R[R["form"].str.startswith("NT 10-K", na=False)].copy()
        late["year"] = late["fdate"].dt.year
        LATE = late.groupby(["cik", "year"], as_index=False).size().rename(columns={"size": "nt_10k"})
        amd = R[R["form"] == "10-K/A"].copy(); amd["year"] = amd["fdate"].dt.year
        AMD = amd.groupby(["cik", "year"], as_index=False).size().rename(columns={"size": "amended_10k"})
        M = LAG.merge(LATE, on=["cik", "year"], how="left").merge(AMD, on=["cik", "year"], how="left")
        M[["nt_10k", "amended_10k"]] = M[["nt_10k", "amended_10k"]].fillna(0)
        P = pd.DataFrame(prof).drop_duplicates("cik")
        P["sic2"] = P["sic"].str[:2]
        META_TBL = M.merge(P, on="cik", how="left")
        META_NOTE.append("FILING METADATA (all mandatory, all already cached):")
        META_NOTE.append(f"  filing lag (days, period end → filing): median={M['filing_lag'].median():.0f} · "
                         f"p10={M['filing_lag'].quantile(.1):.0f} · p90={M['filing_lag'].quantile(.9):.0f} · "
                         f"n={len(M)}")
        META_NOTE.append(f"  NT 10-K late notifications: {int(M['nt_10k'].sum())} firm-years · "
                         f"{int((M.groupby('cik')['nt_10k'].max() > 0).sum())} firms")
        META_NOTE.append(f"  10-K/A amendments:          {int(M['amended_10k'].sum())} firm-years · "
                         f"{int((M.groupby('cik')['amended_10k'].max() > 0).sum())} firms")
        META_NOTE.append(f"  distinct 4-digit SIC codes: {P['sic'].nunique()} · 2-digit groups: {P['sic2'].nunique()}"
                         + (f" (panel 'industry_raw' has {df['industry_raw'].nunique()})"
                            if G("df") is not None and "industry_raw" in df.columns else ""))
        META_NOTE.append(f"  filer categories: {dict(P['filer_category'].value_counts().head(4))}")
        META_NOTE.append(f"  fiscal year ends: {P['fye'].nunique()} distinct — the FILED value, replacing the")
        META_NOTE.append("    month heuristic used in the fiscal-year reconciliation.")
        if G("df") is not None:
            globals()["df"] = df.merge(META_TBL, on=["cik", "year"], how="left")
            for c in ["nt_10k", "amended_10k"]:
                if c in df.columns: df[c] = df[c].fillna(0)
            try:
                import statsmodels.api as sm
                W = df.sort_values(["cik", "year"]).copy()
                lgl = W[["cik", "year", "filing_lag"]].copy(); lgl["year"] += 1
                W = W.merge(lgl.rename(columns={"filing_lag": "lag_L"}), on=["cik", "year"], how="left")
                ctl = [c for c in ["size", "lev", "rnd"] if c in W.columns]
                S = W.dropna(subset=["lag_L", "roa"] + ctl)
                if len(S) > 80 and S["lag_L"].std() > 0:
                    X = S[["lag_L"] + ctl].astype(float)
                    X = pd.concat([X, pd.get_dummies(S["year"].astype(str), prefix="y",
                                                     drop_first=True, dtype=float)], axis=1)
                    icol = "sic2" if "sic2" in S.columns and S["sic2"].nunique() > 3 else "industry_raw"
                    if icol in S.columns and S[icol].nunique() > 1:
                        X = pd.concat([X, pd.get_dummies(S[icol].astype(str), prefix="i",
                                                         drop_first=True, dtype=float)], axis=1)
                    r_ = sm.OLS(S["roa"].values, sm.add_constant(X)).fit(
                        cov_type="cluster", cov_kwds={"groups": S["cik"].values})
                    META_NOTE.append(f"  filing lag(t−1) → ROA (FE on {icol}): β={r_.params['lag_L']:+.5f} "
                                     f"p={r_.pvalues['lag_L']:.3f} n={len(S)}")
                    META_NOTE.append("    NEGATIVE ⇒ slower close predicts weaker performance. Interpret as")
                    META_NOTE.append("    operational capability, not architecture per se — the two overlap,")
                    META_NOTE.append("    and lag also reflects auditor and complexity effects.")
                for c in ["mw_it", "ev_restate_402"]:
                    if c in df.columns and df[c].std() > 0 and df["filing_lag"].notna().any():
                        META_NOTE.append(f"    corr(filing lag, {c}) = {df['filing_lag'].corr(df[c]):+.3f}")
            except Exception as e:
                META_NOTE.append("  outcome test unavailable: " + type(e).__name__)
        try: META_TBL.to_csv(f"{OUT_DIR}/filing_metadata.csv", index=False)
        except Exception: pass
    globals()["META_NOTE"] = META_NOTE
    print("\n".join(META_NOTE) if META_NOTE else "no filing metadata recovered")


In [ ]:
# ============ 5g · CUSTOMISATION VOLUME (XBRL capitalised software) ============
# The 2x2 turns on HOW MUCH a firm customises. The panel measures only how CLEANLY extensions
# are built (adherence), never the volume - so its central question has been untestable.
# The closest public measure is CAPITALISED INTERNAL-USE SOFTWARE (ASC 350-40): money spent
# building software rather than buying it. It is a MANDATORY disclosure when material, which is
# exactly where the protocol says to look. capsoft_intensity exists in the panel at only 13%
# coverage; XBRL company facts carry the same concept for far more firms.
CUST_NOTE, CUST_TBL = [], None
SOFT_TAGS = ["CapitalizedComputerSoftwareNet", "CapitalizedComputerSoftwareGross",
             "CapitalizedComputerSoftwareAdditions", "PaymentsToDevelopSoftware",
             "PaymentsForSoftware", "CapitalizedSoftwareDevelopmentCosts"]
if G("FIRM_CIKS") is None or G("CACHE_DIR") is None:
    print("SKIPPED - no cache.")
else:
    # is there a usable local xbrl store already?
    xdirs = []
    for root, dirs, files in os.walk(os.path.dirname(os.path.dirname(CACHE_DIR))):
        if os.path.basename(root).lower() == "xbrl" and files:
            xdirs.append((root, len(files)))
    if xdirs:
        CUST_NOTE.append(f"local xbrl store found: {xdirs[0][0]} ({xdirs[0][1]} files) - inspect its")
        CUST_NOTE.append("  layout to wire it in; the SEC companyfacts route below runs regardless.")
    cf_path = f"{CACHE_DIR}/capsoft_xbrl.csv"
    cx = read_cache(cf_path, required=["cik", "year", "tag", "val"], label="capsoft_xbrl.csv")
    if cx is not None:
        print(f"Reusing XBRL software facts - {len(cx)} firm-years")
    else:
        recs = []
        for cik in tqdm(FIRM_CIKS, desc="XBRL companyfacts"):
            fp = f"{CACHE_DIR}/facts_{cik}.json"
            try:
                if os.path.exists(fp):
                    j = json.load(open(fp))
                else:
                    j = json.loads(_get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{int(cik):010d}.json"))
                    json.dump({"facts": {"us-gaap": {k: v for k, v in
                               j.get("facts", {}).get("us-gaap", {}).items() if k in SOFT_TAGS}}},
                              open(fp, "w"))     # store only what we need
            except Exception:
                continue
            ug = j.get("facts", {}).get("us-gaap", {})
            for tag in SOFT_TAGS:
                for unit, rows in ug.get(tag, {}).get("units", {}).items():
                    for r in rows:
                        fy = r.get("fy")
                        if fy and YEAR_MIN <= int(fy) <= YEAR_MAX and r.get("form", "").startswith("10-K"):
                            recs.append({"cik": cik, "year": int(fy), "tag": tag, "val": float(r.get("val", 0))})
        cx = (pd.DataFrame(recs).groupby(["cik", "year", "tag"], as_index=False)["val"].max()
              if recs else pd.DataFrame())
        if len(cx): cx.to_csv(cf_path, index=False)
    if len(cx):
        wide = cx.pivot_table(index=["cik", "year"], columns="tag", values="val", aggfunc="max").reset_index()
        have = [t for t in SOFT_TAGS if t in wide.columns]
        wide["capsoft_xbrl"] = wide[have].max(axis=1) if have else np.nan
        CUST_NOTE.append("XBRL capitalised-software coverage:")
        for t in have:
            CUST_NOTE.append(f"    {t:38s} {wide[t].notna().mean():5.1%} of matched firm-years")
        CUST_NOTE.append(f"    ANY tag present: {wide['capsoft_xbrl'].notna().sum()} firm-years · "
                         f"{wide.loc[wide['capsoft_xbrl'].notna(), 'cik'].nunique()} firms")
        if G("df") is not None:
            D2 = df.merge(wide[["cik", "year", "capsoft_xbrl"]], on=["cik", "year"], how="left")
            globals()["df"] = D2
            base_cov = float(df["capsoft"].notna().mean()) if "capsoft" in df.columns else 0.0
            new_cov = float(D2["capsoft_xbrl"].notna().mean())
            CUST_NOTE.append(f"    coverage vs the panel's capsoft_intensity: {base_cov:.0%} → {new_cov:.0%}")
            if new_cov >= 0.30:
                import statsmodels.api as sm
                S = D2.dropna(subset=["capsoft_xbrl", "roa"]).copy()
                # scale by assets when available so it is an INTENSITY, not a level
                if "size" in S.columns:
                    S["cust_int"] = S["capsoft_xbrl"] / np.exp(S["size"]).replace(0, np.nan)
                else:
                    S["cust_int"] = S["capsoft_xbrl"]
                S = S.replace([np.inf, -np.inf], np.nan).dropna(subset=["cust_int"])
                S["cust_int"] = S["cust_int"].clip(S["cust_int"].quantile(.01), S["cust_int"].quantile(.99))
                ctl2 = [c for c in ["size", "lev", "rnd"] if c in S.columns]
                if len(S) > 80:
                    X = S[["cust_int"] + ctl2].astype(float)
                    X = pd.concat([X, pd.get_dummies(S["year"].astype(str), prefix="y",
                                                     drop_first=True, dtype=float)], axis=1)
                    ic = "sic2" if "sic2" in S.columns and S["sic2"].nunique() > 3 else "industry_raw"
                    if ic in S.columns and S[ic].nunique() > 1:
                        X = pd.concat([X, pd.get_dummies(S[ic].astype(str), prefix="i",
                                                         drop_first=True, dtype=float)], axis=1)
                    r_ = sm.OLS(S["roa"].values, sm.add_constant(X)).fit(
                        cov_type="cluster", cov_kwds={"groups": S["cik"].values})
                    CUST_NOTE.append(f"  CUSTOMISATION VOLUME → ROA: β={r_.params['cust_int']:+.4f} "
                                     f"p={r_.pvalues['cust_int']:.3f} n={len(S)}")
                    # THE 2x2's core claim: volume should pay where differentiation potential is high
                    if "rnd" in S.columns:
                        S["dif"] = S.groupby("cik")["rnd"].transform("mean")
                        S["ixn"] = (S["cust_int"] - S["cust_int"].mean()) * (S["dif"] - S["dif"].mean())
                        X2 = pd.concat([X, S[["dif", "ixn"]].astype(float)], axis=1)
                        r2_ = sm.OLS(S["roa"].values, sm.add_constant(X2)).fit(
                            cov_type="cluster", cov_kwds={"groups": S["cik"].values})
                        CUST_NOTE.append(f"  VOLUME × differentiation potential: β={r2_.params['ixn']:+.4f} "
                                         f"p={r2_.pvalues['ixn']:.3f}")
                        CUST_NOTE.append("    POSITIVE ⇒ customisation pays where differentiation is high and")
                        CUST_NOTE.append("    costs where it is not — the 2x2's central claim, finally testable.")
                    CUST_TBL = S[["cik", "year", "cust_int"]]
            else:
                CUST_NOTE.append("    coverage still under 30% — report as a limitation, not a test.")
        try: wide.to_csv(f"{OUT_DIR}/capsoft_xbrl.csv", index=False)
        except Exception: pass
    else:
        CUST_NOTE.append("No XBRL software facts retrieved; customisation VOLUME remains unmeasured.")
    globals()["CUST_NOTE"] = CUST_NOTE
    print("\n".join(CUST_NOTE))


In [ ]:
# ============ 6P1 · CIAOF CORE ANALYSES ============
# Every block below is lifted VERBATIM from the full research pipeline. Nothing is rewritten, so a
# result cannot differ between the lean CIAOF branch and the full branch — only the surrounding
# measurement analyses, which no CIAOF result depends on, are absent.
#
# Contains: adherence variance decomposition; the primary between/within estimate under year and
# industry x year fixed effects; the inverted-U shape test; the alternative-composite robustness
# check; and the 2x2 heterogeneity interaction.
ALT_NOTE, VAR_NOTE = [], []
if G("df") is None or G("CTRL") is None:
    print("SKIPPED — panel or control set unavailable.")
else:
    import statsmodels.api as sm
    A_ = df.copy()
    base_cols = [c for c in ["size", "lev", "rnd"] if c in A_.columns]
    CT = [c + "_L" for c in base_cols]
    lg = A_[["cik", "year", "ci"] + base_cols].copy(); lg["year"] += 1
    lg = lg.rename(columns={c: c + "_L" for c in ["ci"] + base_cols})
    A_ = A_.merge(lg, on=["cik", "year"], how="left")

    # ---------- (5) CORE INTEGRITY AS A BETWEEN-FIRM PROPERTY ----------
    ci_d = df.dropna(subset=["ci"]).copy()
    if len(ci_d) > 60:
        _w = float(ci_d.groupby("cik")["ci"].transform(lambda x: x - x.mean()).std())
        _b = float(ci_d.groupby("cik")["ci"].mean().std())
        _share = _b ** 2 / max(1e-12, _b ** 2 + _w ** 2)
        ALT_NOTE.append(f"[P1] CI variance decomposition: between-firm sd={_b:.4f} · within-firm sd={_w:.4f} "
                        f"· between share of variance={_share:.0%}")
        m2_ = A_.dropna(subset=["ci_L", "roa"] + CT).copy()
        if len(m2_) > 60:
            m2_["ci_bar"] = m2_.groupby("cik")["ci_L"].transform("mean")
            m2_["ci_dev"] = m2_["ci_L"] - m2_["ci_bar"]
            Xc = sm.add_constant(pd.concat([m2_[["ci_bar", "ci_dev"] + CT].astype(float),
                                            pd.get_dummies(m2_["year"].astype(str), prefix="yr",
                                                           drop_first=True, dtype=float)], axis=1))
            rc = sm.OLS(m2_["roa"].values, Xc).fit(cov_type="cluster", cov_kwds={"groups": m2_["cik"].values})
            ALT_NOTE.append(f"[P1] CIAOF (year FE) — CI between beta={rc.params['ci_bar']:+.4f} "
                            f"(p={rc.pvalues['ci_bar']:.3f}) · within beta={rc.params['ci_dev']:+.4f} "
                            f"(p={rc.pvalues['ci_dev']:.3f})")
            # the decisive control: sector composition. A between-firm estimate without industry
            # is a comparison of industries as much as of firms.
            if "industry_raw" in m2_.columns and m2_["industry_raw"].nunique() > 1:
                Xci = sm.add_constant(pd.concat([
                    m2_[["ci_bar", "ci_dev"] + CT].astype(float),
                    pd.get_dummies(m2_["industry_raw"].astype(str) + "_" + m2_["year"].astype(str),
                                   prefix="iy", drop_first=True, dtype=float)], axis=1))
                rci = sm.OLS(m2_["roa"].values, Xci).fit(cov_type="cluster", cov_kwds={"groups": m2_["cik"].values})
                ALT_NOTE.append(f"[P1] CIAOF primary (INDUSTRY x YEAR FE) — CI between "
                                f"beta={rci.params['ci_bar']:+.4f} (p={rci.pvalues['ci_bar']:.3f}) · "
                                f"within beta={rci.params['ci_dev']:+.4f} (p={rci.pvalues['ci_dev']:.3f}) · "
                                f"industries={m2_['industry_raw'].nunique()}")
                _b0, _b1 = float(rc.params["ci_bar"]), float(rci.params["ci_bar"])
                if abs(_b0) > 1e-9:
                    ALT_NOTE.append(f"[P1]   sector composition explains {100*(1-_b1/_b0):.0f}% of the raw "
                                    "between-firm coefficient")
                ALT_NOTE.append(f"[P1]   per-1sd(between) ROA effect with industry FE = "
                                f"{_b1 * float(ci_d.groupby('cik')['ci'].mean().std()):+.4f}")
            # the inverted-U is a SHAPE claim: it can only be traced across the full range,
            # i.e. between firms. Testing it on within-firm variation is near-impossible.
            m2_["ci_bar2"] = m2_["ci_bar"] ** 2
            Xq = sm.add_constant(m2_[["ci_bar", "ci_bar2"] + CT].astype(float))
            rq = sm.OLS(m2_["roa"].values, Xq).fit(cov_type="cluster", cov_kwds={"groups": m2_["cik"].values})
            b1, b2, p2 = rq.params["ci_bar"], rq.params["ci_bar2"], rq.pvalues["ci_bar2"]
            nt = f"inverted-U tested BETWEEN firms: b1={b1:+.4f} b2={b2:+.4f} (p={p2:.3f})"
            if b2 < 0 and p2 < 0.10:
                vx = -b1 / (2 * b2)
                lo, hi = m2_["ci_bar"].quantile([0.05, 0.95])
                inside = bool(lo <= vx <= hi)
                nt += (f" · vertex={vx:+.3f} {'INSIDE' if inside else 'OUTSIDE'} the 5–95% range "
                       f"[{lo:+.3f},{hi:+.3f}]" + ("" if inside else " → NOT an interior optimum; do not report as an inverted-U"))
            else:
                nt += " · not concave at conventional levels → no inverted-U claim"
            ALT_NOTE.append("[P1]   " + nt)
            ALT_NOTE.append("[P1]   CAUTION: between-firm estimates are descriptive. Disciplined firms differ in age,")
            ALT_NOTE.append("[P1]   M&A history, estate complexity and industry; reverse causality (profits fund")
            ALT_NOTE.append("[P1]   discipline) is unresolved. Report as association, not effect.")

    if "ci_alt" in df.columns and df["ci_alt"].notna().sum() > 100:
        a2 = A_.copy()
        lg4 = df[["cik", "year", "ci_alt"]].copy(); lg4["year"] += 1
        a2 = a2.merge(lg4.rename(columns={"ci_alt": "ci_alt_L"}), on=["cik", "year"], how="left")
        a2 = a2.dropna(subset=["ci_alt_L", "roa"] + CT)
        if len(a2) > 60:
            a2["cb"] = a2.groupby("cik")["ci_alt_L"].transform("mean")
            a2["cd"] = a2["ci_alt_L"] - a2["cb"]
            Xa = sm.add_constant(a2[["cb", "cd"] + CT].astype(float))
            ra = sm.OLS(a2["roa"].values, Xa).fit(cov_type="cluster", cov_kwds={"groups": a2["cik"].values})
            ALT_NOTE.append(f"[P1] CI robustness (ccai_pca): between beta={ra.params['cb']:+.4f} "
                            f"(p={ra.pvalues['cb']:.3f}) · within beta={ra.params['cd']:+.4f} (p={ra.pvalues['cd']:.3f})")

    globals()["ALT_NOTE"] = ALT_NOTE
    print("\n".join(ALT_NOTE))

    # ---- 2x2 heterogeneity (verbatim) ----
    D = df.copy()
    ctl = [c for c in ["size", "lev", "rnd"] if c in D.columns]   # control list, as named upstream
    # ---------- (c) 2x2 HETEROGENEITY: does standardisation pay more where differentiation is low? ----------
    if "rnd" in D.columns:
        H = D.dropna(subset=["ci", "roa", "rnd"]).copy()
        H["diff_int"] = H.groupby("cik")["rnd"].transform("mean")       # differentiation intensity proxy
        H["hi_diff"] = (H["diff_int"] > H["diff_int"].median()).astype(float)
        H["ci_b"] = H.groupby("cik")["ci"].transform("mean")
        H["ci_x_diff"] = (H["ci_b"] - H["ci_b"].mean()) * (H["hi_diff"] - H["hi_diff"].mean())
        Xh = H[["ci_b", "hi_diff", "ci_x_diff"] + [c for c in ctl if c in H.columns]].astype(float)
        Xh = pd.concat([Xh, pd.get_dummies(H["year"].astype(str), prefix="y", drop_first=True, dtype=float)], axis=1)
        if "industry_raw" in H.columns and H["industry_raw"].nunique() > 1:
            Xh = pd.concat([Xh, pd.get_dummies(H["industry_raw"].astype(str), prefix="i",
                                               drop_first=True, dtype=float)], axis=1)
        r_ = sm.OLS(H["roa"].values, sm.add_constant(Xh)).fit(cov_type="cluster",
                                                              cov_kwds={"groups": H["cik"].values})
        VAR_NOTE.append(f"2x2 HETEROGENEITY (CI × high-differentiation, R&D-intensity split, n={len(H)}): "
                        f"β_ixn={r_.params['ci_x_diff']:+.4f} p={r_.pvalues['ci_x_diff']:.3f} | "
                        f"β_CI(low-diff)={r_.params['ci_b']:+.4f} (p={r_.pvalues['ci_b']:.3f})")
        VAR_NOTE.append("  the grid predicts β_ixn < 0: standardisation should pay LESS where")
        VAR_NOTE.append("  differentiation intensity is high. This is the 2x2's testable content.")
    globals()["VAR_NOTE"] = VAR_NOTE
    print("\n".join(VAR_NOTE))


In [ ]:
# ============ 6d · H2 WITH A TESTABLE MODERATOR + FORMATIVE-vs-REFLECTIVE ============
# H2 died because EAGI has no reliability. But the moderator does not have to be text.
# The STRUCTURAL governance index (board technology committee + named CIO/CTO/CDO/Chief Architect)
# has real prevalence and is not word-frequency — so H2 becomes testable.
#
# MEASUREMENT-THEORY POINT that matters for BOTH papers:
#   REFLECTIVE construct → a latent trait causes the indicators → they must correlate → α applies.
#   FORMATIVE construct  → the indicators CONSTITUTE the construct → correlation is NOT required
#                          → α is the WRONG statistic.
# EAGI is reflective by design (every term should reflect one governance emphasis) → α=-0.06 is a
# genuine failure. The CIAOF dimensions and the structural index are FORMATIVE (a firm can be high
# on API integration and low on data harmonisation and still be coherently described) → α=0.299 is not a
# defect, it is expected. Formative constructs are validated by criterion/nomological validity and
# must ALSO be reported dimension by dimension.
H2S_NOTE = []
if G("df") is None:
    print("SKIPPED — no panel.")
else:
    import statsmodels.api as sm
    D = df.copy()
    st_cols = [c for c in ["techcom_strict", "techcom", "exec_cio", "exec_cto", "exec_cdo", "exec_arch"]
               if c in D.columns]
    st_cols = [c for c in st_cols if not (c == "techcom" and "techcom_strict" in st_cols)]
    if len(st_cols) >= 2 and "ci" in D.columns:
        D["gov_s"] = D[st_cols].fillna(0).sum(axis=1)
        D = D.sort_values(["cik", "year"])
        lg = D[["cik", "year", "gov_s", "ci"] + [c for c in ["size", "lev", "rnd"] if c in D.columns]].copy()
        lg["year"] += 1
        D = D.merge(lg.rename(columns={c: c + "_L" for c in ["gov_s", "ci"] +
                                       [x for x in ["size", "lev", "rnd"] if x in D.columns]}),
                    on=["cik", "year"], how="left")
        CT2 = [c + "_L" for c in ["size", "lev", "rnd"] if c + "_L" in D.columns]
        H = D.dropna(subset=["gov_s_L", "ci_L", "roa"] + CT2).copy()
        H2S_NOTE.append(f"structural moderator: {'+'.join(st_cols)} · mean={H['gov_s_L'].mean():.2f} "
                        f"· sd={H['gov_s_L'].std():.2f} · n={len(H)}")
        if len(H) > 80:
            H["ci_c"] = H["ci_L"] - H["ci_L"].mean()
            H["gv_c"] = H["gov_s_L"] - H["gov_s_L"].mean()
            H["ixn"] = H["ci_c"] * H["gv_c"]
            for lab, use_ind in [("year FE", False), ("industry+year FE", True)]:
                X = H[["ci_c", "gv_c", "ixn"] + CT2].astype(float)
                X = pd.concat([X, pd.get_dummies(H["year"].astype(str), prefix="y",
                                                 drop_first=True, dtype=float)], axis=1)
                if use_ind and "industry_raw" in H.columns and H["industry_raw"].nunique() > 1:
                    X = pd.concat([X, pd.get_dummies(H["industry_raw"].astype(str), prefix="i",
                                                     drop_first=True, dtype=float)], axis=1)
                r = sm.OLS(H["roa"].values, sm.add_constant(X)).fit(
                    cov_type="cluster", cov_kwds={"groups": H["cik"].values})
                H2S_NOTE.append(f"H2-structural ({lab}): β_ixn={r.params['ixn']:+.4f} p={r.pvalues['ixn']:.3f}"
                                f" | β_CI={r.params['ci_c']:+.4f} (p={r.pvalues['ci_c']:.3f})"
                                f" | β_gov={r.params['gv_c']:+.4f} (p={r.pvalues['gv_c']:.3f})")
            H2S_NOTE.append("  reading: β_ixn > 0 ⇒ adherence pays MORE where governance capacity exists"
                            " (the 'governed capacity to choose' proposition).")
            H2S_NOTE.append("  NOTE: the structural index is FORMATIVE — internal consistency (α) is not")
            H2S_NOTE.append("  the relevant validity test; criterion validity and dimension-level results are.")

    # ---- CIAOF dimension-by-dimension (required for a formative composite) ----
    kpis = [c for c in df.columns if str(c).startswith("kpi_") or str(c).endswith("_kpi")]
    if not kpis and G("panel") is not None:
        kpis = [c for c in panel.columns if str(c).endswith("_kpi")]
    if kpis:
        rows = []
        B = df.dropna(subset=["roa"]).copy()
        for k in kpis:
            if k not in B.columns: continue
            sub = B.dropna(subset=[k])
            if len(sub) < 80: continue
            sub = sub.copy()
            sub["kb"] = sub.groupby("cik")[k].transform("mean")
            X = sub[["kb"] + [c for c in ["size", "lev", "rnd"] if c in sub.columns]].astype(float)
            X = pd.concat([X, pd.get_dummies(sub["year"].astype(str), prefix="y", drop_first=True, dtype=float)], axis=1)
            if "industry_raw" in sub.columns and sub["industry_raw"].nunique() > 1:
                X = pd.concat([X, pd.get_dummies(sub["industry_raw"].astype(str), prefix="i",
                                                 drop_first=True, dtype=float)], axis=1)
            r = sm.OLS(sub["roa"].values, sm.add_constant(X)).fit(
                cov_type="cluster", cov_kwds={"groups": sub["cik"].values})
            # Raw-scale coefficients across dimensions are not comparable to each other or to the
            # composite: each KPI has its own dispersion. Report per-1sd(between) alongside the raw
            # coefficient so magnitudes are interpretable and implausible values are visible.
            _sdb = float(sub.groupby("cik")[k].mean().std())
            rows.append(dict(dimension=k, beta_between=round(float(r.params["kb"]), 4),
                             sd_between=round(_sdb, 4),
                             per_1sd=round(float(r.params["kb"]) * _sdb, 4),
                             p=round(float(r.pvalues["kb"]), 3), n=len(sub)))
        if rows:
            DIMS = pd.DataFrame(rows).sort_values("p")
            globals()["DIMS"] = DIMS
            H2S_NOTE.append("CIAOF dimension-level (between-firm, industry+year FE) — per_1sd is the")
            H2S_NOTE.append("interpretable magnitude; raw betas are on each dimension's own scale — required because")
            H2S_NOTE.append("the composite is formative and may average out opposing dimension effects:")
            for ln in DIMS.to_string(index=False).splitlines(): H2S_NOTE.append("  " + ln)
            try: DIMS.to_csv(f"{OUT_DIR}/ciaof_dimension_level.csv", index=False)
            except Exception: pass
    globals()["H2S_NOTE"] = H2S_NOTE
    print("\n".join(H2S_NOTE) if H2S_NOTE else "No structural moderator or KPI dimensions available.")


In [ ]:
# ============ 6g · P1-H3 · ALLOCATION, NOT LEVEL ============
# P1-H2 (inverted-U) asked whether there is an optimal LEVEL of adherence. Rejected.
# P1-H3 asks a different question: at a GIVEN level, does SELECTIVE adherence beat UNIFORM
# adherence? The practitioner claim behind the 2x2 is that standard-everywhere and
# custom-everywhere are both corner solutions, and that the payoff comes from standardising
# supporting capabilities while differentiating revenue-critical ones.
#
#   level      = mean across the five CIAOF dimensions   (how much adherence)
#   dispersion = SD across the five dimensions           (how SELECTIVELY it is applied)
#
# MECHANICAL CAVEAT: with bounded indicators, dispersion is constrained by the level (a firm at
# 0.95 cannot be as dispersed as one at 0.50). Dispersion is therefore residualised on level and
# level-squared before use, so the coefficient is not a repackaged level effect.
H3_NOTE = []
kpic = [c for c in (df.columns if G("df") is not None else []) if str(c).startswith("kpi_")]
if G("df") is None or len(kpic) < 3:
    print("SKIPPED — fewer than three CIAOF dimensions in the panel.")
else:
    import statsmodels.api as sm
    D = df.copy()
    K = D[kpic].astype(float)
    Kz = (K - K.mean()) / K.std().replace(0, 1)          # comparable units across dimensions
    D["ci_level"] = Kz.mean(axis=1)
    D["ci_disp_raw"] = Kz.std(axis=1)
    S = D.dropna(subset=["ci_level", "ci_disp_raw", "roa"]).copy()
    H3_NOTE.append(f"dimensions used (k={len(kpic)}): {[c.replace('kpi_','') for c in kpic]}")
    if len(S) > 100:
        Xr = sm.add_constant(pd.DataFrame({"l": S["ci_level"], "l2": S["ci_level"] ** 2}))
        S["ci_disp"] = S["ci_disp_raw"].values - sm.OLS(S["ci_disp_raw"].values, Xr).fit().predict(Xr)
        H3_NOTE.append(f"dispersion residualised on level and level² · "
                       f"corr(level, raw disp)={S['ci_level'].corr(S['ci_disp_raw']):+.3f} → "
                       f"corr(level, residual disp)={S['ci_level'].corr(S['ci_disp']):+.3f}")
        ctl = [c for c in ["size", "lev", "rnd"] if c in S.columns]
        def blk(SS, cols):
            X = SS[cols + ctl].astype(float)
            X = pd.concat([X, pd.get_dummies(SS["year"].astype(str), prefix="y", drop_first=True, dtype=float)], axis=1)
            if "industry_raw" in SS.columns and SS["industry_raw"].nunique() > 1:
                X = pd.concat([X, pd.get_dummies(SS["industry_raw"].astype(str), prefix="i",
                                                 drop_first=True, dtype=float)], axis=1)
            return sm.OLS(SS["roa"].values, sm.add_constant(X)).fit(
                cov_type="cluster", cov_kwds={"groups": SS["cik"].values})
        # between-firm (the theoretically primary estimator for a firm characteristic)
        B = S.copy()
        B["lvl_b"] = B.groupby("cik")["ci_level"].transform("mean")
        B["dsp_b"] = B.groupby("cik")["ci_disp"].transform("mean")
        r = blk(B, ["lvl_b", "dsp_b"])
        H3_NOTE.append(f"P1-H3 between-firm: β_level={r.params['lvl_b']:+.4f} (p={r.pvalues['lvl_b']:.3f}) · "
                       f"β_DISPERSION={r.params['dsp_b']:+.4f} (p={r.pvalues['dsp_b']:.3f}) n={len(B)}")
        # is there an interior optimum in SELECTIVITY (the shape claim, moved to the right variable)?
        B["dsp_b2"] = B["dsp_b"] ** 2
        r2 = blk(B, ["lvl_b", "dsp_b", "dsp_b2"])
        b1, b2, p2 = r2.params["dsp_b"], r2.params["dsp_b2"], r2.pvalues["dsp_b2"]
        nt = f"P1-H3 shape in dispersion: b1={b1:+.4f} b2={b2:+.4f} (p={p2:.3f})"
        if b2 < 0 and p2 < 0.10:
            vx = -b1 / (2 * b2); lo, hi = B["dsp_b"].quantile([0.05, 0.95])
            nt += (f" · vertex={vx:+.3f} {'INSIDE' if lo <= vx <= hi else 'OUTSIDE'} the 5–95% range"
                   + ("" if lo <= vx <= hi else " → not an interior optimum"))
        else:
            nt += " · not concave → no interior optimum in selectivity"
        H3_NOTE.append(nt)
        # corner comparison: the 2x2 in its own terms
        B["hl"] = (B["lvl_b"] > B["lvl_b"].median()).map({True: "high-adherence", False: "low-adherence"})
        B["hd"] = (B["dsp_b"] > B["dsp_b"].median()).map({True: "SELECTIVE", False: "uniform"})
        cell_ = B.groupby(["hl", "hd"])["roa"].agg(["mean", "count"]).round(4)
        H3_NOTE.append("corner comparison (raw means — descriptive only, no controls):")
        for ln in cell_.to_string().splitlines(): H3_NOTE.append("  " + ln)
        # does selectivity pay MORE where there is more to differentiate on?
        if "rnd" in B.columns:
            B["dif"] = B.groupby("cik")["rnd"].transform("mean")
            B["dsp_x_dif"] = (B["dsp_b"] - B["dsp_b"].mean()) * (B["dif"] - B["dif"].mean())
            r3 = blk(B, ["lvl_b", "dsp_b", "dif", "dsp_x_dif"])
            H3_NOTE.append(f"P1-H3 interaction (dispersion × differentiation potential): "
                           f"β={r3.params['dsp_x_dif']:+.4f} (p={r3.pvalues['dsp_x_dif']:.3f})")
            H3_NOTE.append("  positive ⇒ selectivity pays more where there is more to differentiate on —")
            H3_NOTE.append("  the empirical content of the matrix, and the case for an architect in the loop.")
        # ---------- P1-H3c · IS MORE ALWAYS BETTER? A BIN TEST WITH NO FUNCTIONAL FORM ----------
        # The inverted-U asked this through a quadratic, and a quadratic vertex is an algebraic
        # artefact as often as an optimum. Bins answer it directly: estimate an adjusted mean per
        # adherence quintile and see whether the TOP quintile is actually the best one.
        # NOTE ON REACH: this can only speak to the range the data contains. If no firm sits near
        # the ceiling, nothing here licenses a claim about 100%.
        try:
            # qcut assigns Q1 to the LOWEST quintile. Label by direction so the reference
            # group cannot be misread as "top adherence".
            _ql = ["Q1_lowest_adh", "Q2", "Q3", "Q4", "Q5_highest_adh"]
            B["q5"] = pd.qcut(B["lvl_b"], 5, labels=_ql, duplicates="drop").astype(str)
            if B["q5"].nunique() >= 4:
                Xq = pd.get_dummies(B["q5"], prefix="q", drop_first=True, dtype=float)
                Xq = pd.concat([Xq, B[ctl].astype(float)], axis=1)
                Xq = pd.concat([Xq, pd.get_dummies(B["year"].astype(str), prefix="y", drop_first=True, dtype=float)], axis=1)
                if "industry_raw" in B.columns and B["industry_raw"].nunique() > 1:
                    Xq = pd.concat([Xq, pd.get_dummies(B["industry_raw"].astype(str), prefix="i",
                                                       drop_first=True, dtype=float)], axis=1)
                rq = sm.OLS(B["roa"].values, sm.add_constant(Xq)).fit(
                    cov_type="cluster", cov_kwds={"groups": B["cik"].values})
                base_q = sorted(B["q5"].unique())[0]
                eff = {base_q: 0.0}
                for qq in sorted(B["q5"].unique())[1:]:
                    kx = "q_" + qq
                    if kx in rq.params.index: eff[qq] = float(rq.params[kx])
                best = max(eff, key=eff.get); top = sorted(eff)[-1]
                H3_NOTE.append("P1-H3c adherence QUINTILES (adjusted ROA vs " + base_q +
                               ", industry+year FE) — Q1 = LOWEST adherence, Q5 = HIGHEST:")
                for qq in sorted(eff):
                    pv = rq.pvalues.get("q_" + qq, float("nan"))
                    H3_NOTE.append(f"    {qq}: {eff[qq]:+.4f}" + (f" (p={pv:.3f})" if pv == pv else " (reference)")
                                   + ("   ← highest" if qq == best else ""))
                H3_NOTE.append(f"  observed adherence range: {B['lvl_b'].min():+.2f} to {B['lvl_b'].max():+.2f} "
                               f"(z units) — claims are limited to this range.")
                _nsig = sum(1 for qq in eff if qq != base_q and rq.pvalues.get("q_" + qq, 1) < 0.0125)
                H3_NOTE.append(f"  multiplicity: 4 contrasts vs the reference → Bonferroni 5% is p<0.0125; "
                               f"{_nsig}/4 survive.")
                if _nsig == 0:
                    H3_NOTE.append("  ⇒ NO contrast survives adjustment: quintile differences are small and")
                    H3_NOTE.append("    non-monotonic. Report as 'no reliable gradient', not as a shape claim.")
                if best != top:
                    H3_NOTE.append(f"  ⇒ the HIGHEST-adherence quintile is not the best ({best} is). Within the")
                    H3_NOTE.append("    more adherence is not monotonically better — and this is shown WITHOUT")
                    H3_NOTE.append("    imposing a quadratic, so it is not a vertex artefact.")
                else:
                    H3_NOTE.append("  ⇒ the top quintile IS the best: no evidence against 'more is better' here.")
        except Exception as e:
            H3_NOTE.append("quintile test unavailable: " + type(e).__name__)

        # ---------- P1-H3b · DIRECTIONAL pattern, not just dispersion ----------
        # Core-integrity logic does not say "deviate somewhere". It says deviation is LEGITIMATE on the
        # process dimension (that is where differentiation lives) and DEBT on the four technical
        # hygiene dimensions. So the predicted profile is: high technical hygiene, selectively lower
        # process standardisation. PRE-SPECIFIED, and sensitivity-tested by swapping the roles.
        PROC_DIM = [c for c in kpic if "standard" in c.lower()]
        HYG_DIM = [c for c in kpic if c not in PROC_DIM]
        if len(PROC_DIM) == 1 and len(HYG_DIM) >= 3:
            Z2 = Kz.copy()
            B["hyg"] = Z2[HYG_DIM].mean(axis=1).groupby(D["cik"]).transform("mean").reindex(B.index)
            B["prc"] = Z2[PROC_DIM[0]].groupby(D["cik"]).transform("mean").reindex(B.index)
            B2 = B.dropna(subset=["hyg", "prc"]).copy()
            if len(B2) > 100:
                B2["pattern"] = B2["hyg"] - B2["prc"]          # high hygiene, low process lock-in
                r4 = blk(B2, ["lvl_b", "pattern"])
                H3_NOTE.append(f"P1-H3b DIRECTIONAL pattern (technical hygiene minus process standardisation): "
                               f"β={r4.params['pattern']:+.4f} (p={r4.pvalues['pattern']:.3f}) n={len(B2)}")
                r5 = blk(B2, ["hyg", "prc"])
                H3_NOTE.append(f"  decomposed: β_hygiene={r5.params['hyg']:+.4f} (p={r5.pvalues['hyg']:.3f}) · "
                               f"β_process_std={r5.params['prc']:+.4f} (p={r5.pvalues['prc']:.3f})")
                H3_NOTE.append("  predicted: β_hygiene > 0 (technical deviation is debt) and β_process_std ≤ 0")
                H3_NOTE.append("  (process deviation is where differentiation legitimately lives).")
                # SENSITIVITY: swap the roles. If the reversed pattern performs just as well, the
                # dimension-to-role assignment is doing no work and the test is uninformative.
                B2["pattern_rev"] = B2["prc"] - B2["hyg"]
                r6 = blk(B2, ["lvl_b", "pattern_rev"])
                H3_NOTE.append(f"  SENSITIVITY (roles reversed): β={r6.params['pattern_rev']:+.4f} "
                               f"(p={r6.pvalues['pattern_rev']:.3f}) — by construction the mirror image; "
                               "report both so the theoretical assignment is visible, not hidden.")
        # ---------- P1-H3d · EACH FIRM'S OWN 100% — FIT TO POTENTIAL, NOT ABSOLUTE LEVEL ----------
        # An absolute optimum assumes every firm faces the same feasible maximum. They do not:
        # a firm whose portfolio is 40% genuinely differentiating cannot and should not reach the
        # adherence of one that is 5% differentiating. Both may sit exactly at THEIR optimum while
        # occupying different absolute levels — and pooling them MECHANICALLY FLATTENS the curve.
        # So the variable of interest is the GAP between actual adherence and the level the firm's
        # own business model implies. Predicted optimum: gap = 0, not level = maximum.
        try:
            P = B.dropna(subset=["lvl_b"]).copy()
            prox = [c for c in ["rnd", "size", "lev"] if c in P.columns]     # differentiation potential
            if len(prox) >= 2 and len(P) > 100:
                Xp = P[prox].astype(float)
                if "industry_raw" in P.columns and P["industry_raw"].nunique() > 1:
                    Xp = pd.concat([Xp, pd.get_dummies(P["industry_raw"].astype(str), prefix="p",
                                                       drop_first=True, dtype=float)], axis=1)
                Xp = sm.add_constant(Xp)
                fitm = sm.OLS(P["lvl_b"].values, Xp).fit()
                P["expected_adh"] = fitm.predict(Xp)          # what this firm's model implies
                P["gap"] = P["lvl_b"] - P["expected_adh"]      # + = over-adhering, - = under-adhering
                P["gap2"] = P["gap"] ** 2
                P["absgap"] = P["gap"].abs()
                H3_NOTE.append(f"P1-H3d FIT-TO-POTENTIAL · expected adherence modelled from "
                               f"{prox} + industry (R²={fitm.rsquared:.3f})")
                rg = blk(P, ["gap", "gap2"])
                b1g, b2g, p2g = rg.params["gap"], rg.params["gap2"], rg.pvalues["gap2"]
                nt2 = f"  quadratic in the GAP: b1={b1g:+.4f} b2={b2g:+.4f} (p={p2g:.3f})"
                if b2g < 0 and p2g < 0.10:
                    vx = -b1g / (2 * b2g)
                    nt2 += f" · peak at gap={vx:+.3f} (predicted ≈ 0 ⇒ being at YOUR OWN level is what pays)"
                else:
                    nt2 += " · not concave → no firm-specific optimum detected"
                H3_NOTE.append(nt2)
                ra = blk(P, ["absgap"])
                H3_NOTE.append(f"  distance from own level: β_|gap|={ra.params['absgap']:+.4f} "
                               f"(p={ra.pvalues['absgap']:.3f}) — NEGATIVE ⇒ deviating in EITHER direction costs")
                P["gq"] = pd.qcut(P["gap"], 5, labels=["most under", "under", "at level", "over", "most over"],
                                  duplicates="drop").astype(str)
                gm = P.groupby("gq")["roa"].agg(["mean", "count"]).round(4)
                H3_NOTE.append("  raw ROA by gap quintile (descriptive, no controls):")
                for ln in gm.to_string().splitlines(): H3_NOTE.append("    " + ln)
                # ATYPICALITY PLACEBO: |gap| is a residual, so firms that observables predict
                # POORLY have large |gap| by construction — and atypical firms (conglomerates,
                # restructurings) may perform differently for reasons unrelated to architecture.
                # Build the identical gap from a measure that should carry no information.
                if "placebo" in B.columns:
                    P["plc_b"] = B["placebo"].groupby(B["cik"]).transform("mean").reindex(P.index)
                    Pp = P.dropna(subset=["plc_b"]).copy()
                    if len(Pp) > 100:
                        fitp = sm.OLS(Pp["plc_b"].values, sm.add_constant(Pp[prox].astype(float))).fit()
                        Pp["gap_plc"] = Pp["plc_b"] - fitp.predict(sm.add_constant(Pp[prox].astype(float)))
                        Pp["absgap_plc"] = Pp["gap_plc"].abs()
                        rp = blk(Pp, ["absgap_plc"])
                        H3_NOTE.append(f"  ATYPICALITY PLACEBO: β_|placebo gap|={rp.params['absgap_plc']:+.4f} "
                                       f"(p={rp.pvalues['absgap_plc']:.3f})")
                        if rp.pvalues["absgap_plc"] < 0.10:
                            H3_NOTE.append("    ⚠ a meaningless measure's gap ALSO predicts performance ⇒ |gap| is")
                            H3_NOTE.append("      capturing ATYPICALITY, not architectural misfit. Do not interpret H3d.")
                        else:
                            H3_NOTE.append("    ✓ placebo gap is null ⇒ the misfit reading survives the atypicality critique.")
                H3_NOTE.append("  NOTE: an absolute-scale null is NOT evidence against a firm-specific optimum;")
                H3_NOTE.append("  pooling firms with different feasible maxima flattens the curve by construction.")
                H3_NOTE.append("  CIRCULARITY WARNING: 'expected' adherence is fitted from observables, so the gap")
                H3_NOTE.append("  is a residual, not a normative target. It shows dispersion around peer-typical")
                H3_NOTE.append("  behaviour — it cannot say what a firm SHOULD do. Only the workshops can.")
        except Exception as e:
            H3_NOTE.append("fit-to-potential test unavailable: " + type(e).__name__)

        H3_NOTE.append("CAVEAT: even H3b is a DIMENSION-level proxy. 'Differentiating on the right PROCESSES'")
        H3_NOTE.append("needs firm × process × treatment data, which no public source provides. That is the")
        H3_NOTE.append("data the TRVA workshops generate — the tool is the missing instrument, not a by-product.")
    globals()["H3_NOTE"] = H3_NOTE
    print("\n".join(H3_NOTE))


In [ ]:
# ============ 6h · STRUCTURE OF THE COMPOSITE — four questions H3 does not answer ============
# H3 asked whether SPREAD pays. Four further structural questions, each with a different
# practitioner consequence, and two of them make OPPOSITE predictions to H3:
#   A WEAKEST LINK   is the binding constraint the MINIMUM dimension rather than the mean?
#                    → justifies a floor guardrail ("no dimension below X"), not an aggregate target
#   B COMPLEMENTARITY do dimensions REINFORCE each other? If so, spread HURTS — the direct
#                    competitor to H3. Running both is a horse race, not a confirmation exercise.
#   C IMPORTANCE     Shapley decomposition of R² gives each dimension a defensible % weight,
#                    replacing equal weighting. Gated: decomposing a null model decomposes noise.
#   D NECESSITY      is a dimension a PREREQUISITE (high outcome impossible without it) rather
#                    than a contributor? Necessity and sufficiency are different claims.
STRUCT_NOTE = []
kpic2 = [c for c in (df.columns if G("df") is not None else []) if str(c).startswith("kpi_")]
if G("df") is None or len(kpic2) < 3:
    print("SKIPPED — dimensions unavailable.")
else:
    import statsmodels.api as sm
    from itertools import combinations
    from math import factorial   # np.math was removed in NumPy 2
    D = df.copy()
    K = D[kpic2].astype(float); Kz = (K - K.mean()) / K.std().replace(0, 1)
    for c in kpic2: D["z_" + c] = Kz[c]
    zc = ["z_" + c for c in kpic2]
    F = D.groupby("cik")[zc + ["roa"] + [x for x in ["size", "lev", "rnd"] if x in D.columns]].mean().dropna()
    if "industry_raw" in D.columns:
        F = F.join(D.groupby("cik")["industry_raw"].first())
    ctl = [c for c in ["size", "lev", "rnd"] if c in F.columns]
    def design(S, cols):
        X = S[cols + ctl].astype(float)
        if "industry_raw" in S.columns and S["industry_raw"].nunique() > 1:
            X = pd.concat([X, pd.get_dummies(S["industry_raw"].astype(str), prefix="i",
                                             drop_first=True, dtype=float)], axis=1)
        return sm.add_constant(X)
    def r2(S, cols):
        X = design(S, cols).values; y = S["roa"].values
        b = np.linalg.pinv(X.T @ X) @ X.T @ y; e = y - X @ b
        return 1 - float(e @ e) / float(((y - y.mean()) ** 2).sum())
    if len(F) > 60:
        F["mean_d"] = F[zc].mean(axis=1); F["min_d"] = F[zc].min(axis=1)
        F["max_d"] = F[zc].max(axis=1)      # CEILING: is performance driven by the BEST dimension?
        # A · weakest link vs mean
        rm = sm.OLS(F["roa"].values, design(F, ["mean_d"])).fit(cov_type="HC1")
        rn = sm.OLS(F["roa"].values, design(F, ["min_d"])).fit(cov_type="HC1")
        rb = sm.OLS(F["roa"].values, design(F, ["mean_d", "min_d"])).fit(cov_type="HC1")
        STRUCT_NOTE.append(f"A WEAKEST LINK · mean-only β={rm.params['mean_d']:+.4f} (p={rm.pvalues['mean_d']:.3f}) · "
                           f"min-only β={rn.params['min_d']:+.4f} (p={rn.pvalues['min_d']:.3f})")
        STRUCT_NOTE.append(f"   both entered: β_mean={rb.params['mean_d']:+.4f} (p={rb.pvalues['mean_d']:.3f}) · "
                           f"β_min={rb.params['min_d']:+.4f} (p={rb.pvalues['min_d']:.3f})")
        rx = sm.OLS(F["roa"].values, design(F, ["max_d"])).fit(cov_type="HC1")
        rt = sm.OLS(F["roa"].values, design(F, ["mean_d", "min_d", "max_d"])).fit(cov_type="HC1")
        STRUCT_NOTE.append(f"A2 CEILING · max-only β={rx.params['max_d']:+.4f} (p={rx.pvalues['max_d']:.3f})")
        STRUCT_NOTE.append(f"   all three entered: mean={rt.params['mean_d']:+.4f} (p={rt.pvalues['mean_d']:.3f}) · "
                           f"min={rt.params['min_d']:+.4f} (p={rt.pvalues['min_d']:.3f}) · "
                           f"max={rt.params['max_d']:+.4f} (p={rt.pvalues['max_d']:.3f})")
        _c = F[["mean_d", "min_d", "max_d"]].corr()
        STRUCT_NOTE.append(f"   ⚠ mean/min/max are ORDER STATISTICS OF THE SAME FIVE NUMBERS "
                           f"(corr mean-min={_c.loc['mean_d','min_d']:+.2f}, mean-max={_c.loc['mean_d','max_d']:+.2f}). "
                           "Entering all three")
        STRUCT_NOTE.append("     together is near-collinear: signs can flip between the single-regressor and")
        STRUCT_NOTE.append("     joint specifications. Read the SINGLE-regressor results as primary and treat the")
        STRUCT_NOTE.append("     joint decomposition as indicative only.")
        STRUCT_NOTE.append("   MIN wins  → a FLOOR binds: enforce a minimum per dimension, not an aggregate target.")
        STRUCT_NOTE.append("   MAX wins  → a CEILING logic: one strong dimension carries the firm; the rest are")
        STRUCT_NOTE.append("               hygiene and need only be adequate.")
        STRUCT_NOTE.append("   MEAN wins → dimensions are substitutable and an aggregate score is defensible.")
        STRUCT_NOTE.append("   all null  → the composite has no internal structure worth encoding in the tool.")
        # B · complementarity horse race against H3
        pairs = list(combinations(zc, 2))[:10]
        for a_, b_ in pairs:
            F[f"x_{a_}_{b_}"] = (F[a_] - F[a_].mean()) * (F[b_] - F[b_].mean())
        ixn = [f"x_{a_}_{b_}" for a_, b_ in pairs]
        rc = sm.OLS(F["roa"].values, design(F, zc + ixn)).fit(cov_type="HC1")
        pos = int(sum(1 for k in ixn if rc.params[k] > 0)); sig = int(sum(1 for k in ixn if rc.pvalues[k] < 0.05))
        STRUCT_NOTE.append(f"B COMPLEMENTARITY · {len(ixn)} pairwise interactions: {pos} positive, {sig} significant at 5%")
        STRUCT_NOTE.append("   mostly POSITIVE ⇒ dimensions reinforce each other ⇒ spread HURTS (contradicts H3);")
        STRUCT_NOTE.append("   mostly null ⇒ dimensions are separable ⇒ selective allocation is coherent.")
        # C · Shapley importance weights (gated on the joint model being informative)
        full = r2(F, zc); basec = r2(F, [])
        if full - basec > 0.01:
            sh = {c: 0.0 for c in zc}
            for c in zc:
                others = [x for x in zc if x != c]
                for k in range(len(others) + 1):
                    for sub in combinations(others, k):
                        w = factorial(k) * factorial(len(zc) - k - 1) / factorial(len(zc))
                        sh[c] += w * (r2(F, list(sub) + [c]) - r2(F, list(sub)))
            tot = sum(max(v, 0) for v in sh.values()) or 1.0
            STRUCT_NOTE.append(f"C IMPORTANCE (Shapley share of explained variance; model R²={full:.3f}):")
            for c, v in sorted(sh.items(), key=lambda t: -t[1]):
                STRUCT_NOTE.append(f"     {c.replace('z_kpi_','').replace('_kpi',''):26s} {max(v,0)/tot:6.1%}")
            STRUCT_NOTE.append("   these are DEFENSIBLE weights to replace equal weighting — but they are")
            STRUCT_NOTE.append("   sample-specific and must be reported as such, never as universal constants.")
        else:
            STRUCT_NOTE.append(f"C IMPORTANCE · skipped: dimensions jointly explain only {full - basec:.3f} of "
                               "variance. Decomposing a null model decomposes noise.")
        # D · necessity (prerequisite) vs contribution
        hi = F[F["roa"] >= F["roa"].quantile(0.75)]
        STRUCT_NOTE.append("D NECESSITY · 5th percentile of each dimension, all firms vs HIGH-ROA firms:")
        for c in zc:
            a_all = float(F[c].quantile(0.05)); a_hi = float(hi[c].quantile(0.05))
            flag = "  ← acts as a FLOOR" if (a_hi - a_all) > 0.5 else ""
            STRUCT_NOTE.append(f"     {c.replace('z_kpi_','').replace('_kpi',''):26s} all={a_all:+.2f} "
                               f"high-ROA={a_hi:+.2f} lift={a_hi - a_all:+.2f}{flag}")
        STRUCT_NOTE.append("   a large lift means high performers are never found below that level ⇒ necessary")
        STRUCT_NOTE.append("   condition (prerequisite), which is a different claim from average contribution.")
    # ---- transfer the same battery to the STRUCTURAL governance index ----
    st = [c for c in ["techcom_strict", "exec_cio", "exec_cto", "exec_cdo", "exec_arch"] if c in D.columns]
    if len(st) >= 3:
        G2 = D.groupby("cik")[st + ["roa"] + ctl].mean().dropna()
        if "industry_raw" in D.columns: G2 = G2.join(D.groupby("cik")["industry_raw"].first())
        if len(G2) > 60:
            G2["mean_g"] = G2[st].mean(axis=1); G2["min_g"] = G2[st].min(axis=1)
            ra = sm.OLS(G2["roa"].values, design(G2, ["mean_g"])).fit(cov_type="HC1")
            rb2 = sm.OLS(G2["roa"].values, design(G2, ["min_g"])).fit(cov_type="HC1")
            STRUCT_NOTE.append(f"TRANSFER to structural governance · mean β={ra.params['mean_g']:+.4f} "
                               f"(p={ra.pvalues['mean_g']:.3f}) · min β={rb2.params['min_g']:+.4f} "
                               f"(p={rb2.pvalues['min_g']:.3f})")
            # FIT-TO-POTENTIAL for governance: the claim is NOT "more governance is better".
            # It is that governance should be PROPORTIONATE TO ARCHITECTURAL COMPLEXITY.
            # Over-governing a simple firm is bureaucracy; under-governing a complex one is chaos.
            cx = [c for c in ["size", "lev", "rnd", "aux_n_tokens"] if c in D.columns]
            if len(cx) >= 2:
                CX = D.groupby("cik")[cx].mean()
                G3 = G2.join(CX, how="inner", rsuffix="_cx").dropna()
                if len(G3) > 60:
                    xcols = [c for c in G3.columns if c in cx or c.endswith("_cx")]
                    Xc = sm.add_constant(G3[xcols].astype(float))
                    fg = sm.OLS(G3["mean_g"].values, Xc).fit()
                    G3["gap_g"] = G3["mean_g"] - fg.predict(Xc)
                    G3["absgap_g"] = G3["gap_g"].abs()
                    rg1 = sm.OLS(G3["roa"].values, design(G3, ["absgap_g"])).fit(cov_type="HC1")
                    STRUCT_NOTE.append(f"   FIT-TO-POTENTIAL on governance (complexity proxies {xcols}, "
                                       f"R²={fg.rsquared:.3f}): β_|gap|={rg1.params['absgap_g']:+.4f} "
                                       f"(p={rg1.pvalues['absgap_g']:.3f})")
                    STRUCT_NOTE.append("   NEGATIVE ⇒ governance mismatched to complexity costs in BOTH directions:")
                    STRUCT_NOTE.append("   bureaucracy in simple firms, chaos in complex ones. That is a sharper claim")
                    STRUCT_NOTE.append("   than 'more governance is better' and it is what the tool should encode.")
            STRUCT_NOTE.append("   the EAGI TEXT index cannot carry this framing: a gap between two noise draws")
            STRUCT_NOTE.append("   is still noise (α=-0.06). Fit-to-potential transfers to STRUCTURE, not to text.")
            STRUCT_NOTE.append("   the text index cannot support dimension analysis (terms do not co-move);")
            STRUCT_NOTE.append("   the structural index can, and that is where the EAGI transfer lands.")
    globals()["STRUCT_NOTE"] = STRUCT_NOTE
    print("\n".join(STRUCT_NOTE))


In [ ]:
# ============ 9b · FALSIFICATION & SPECIFICATION CURVE ============
# Two things a tier-1 referee will demand of results this fragile:
#   (a) a PLACEBO run of every firm-level test — if meaningless vocabulary "predicts" the same
#       outcomes, the finding is a document/firm property, not the construct;
#   (b) a SPECIFICATION CURVE — when significance flips across reasonable specs, the honest
#       object is the distribution of estimates, not a chosen point estimate.
FALSI, SPECCURVE = None, None
if G("df") is None:
    print("SKIPPED — no panel.")
else:
    import statsmodels.api as sm
    D = df.copy()
    idxc = ("eagi_best" if "eagi_best" in D.columns and D["eagi_best"].notna().sum() > 100
            else ("eagi" if "eagi" in D.columns else
                  ("ci" if "ci" in D.columns else None)))
    if idxc == "ci":
        print("  no text index in this branch — the falsification battery runs on the CI index,")
        print("  which is the construct the CIAOF paper reports.")
    if idxc is None:
        globals()["SPECCURVE"] = None
        print("  text index absent — CIAOF blocks only")

    # ---------- (a) PLACEBO falsification of the firm-level results ----------
    if "placebo" in D.columns:
        ad = D.dropna(subset=["roa"]).sort_values(["cik", "year"])
        gg = ad.groupby("cik")
        F = pd.DataFrame({"vol": gg["roa"].std(),
                          "worst": gg["roa"].apply(lambda x: float(x.diff().min()) if len(x) > 2 else np.nan),
                          "real": gg[idxc].mean(), "plac": gg["placebo"].mean()}).reset_index()
        for c in ["size", "lev", "rnd"]:
            if c in ad.columns: F[c] = gg[c].mean().values
        if "industry_raw" in ad.columns: F["ind"] = gg["industry_raw"].first().values
        F = F.dropna(subset=["vol", "worst", "real", "plac"])
        if len(F) > 40:
            base = [c for c in ["size", "lev", "rnd"] if c in F.columns]
            rows = []
            for dep in ["vol", "worst"]:
                for xv, lab in [("real", f"{idxc} (construct)"), ("plac", "PLACEBO lexicon")]:
                    X = F[[xv] + base].astype(float)
                    if "ind" in F.columns and F["ind"].nunique() > 1:
                        X = pd.concat([X, pd.get_dummies(F["ind"].astype(str), prefix="i",
                                                         drop_first=True, dtype=float)], axis=1)
                    r = sm.OLS(F[dep].values, sm.add_constant(X)).fit(cov_type="HC1")
                    rows.append(dict(outcome=dep, regressor=lab, beta=round(float(r.params[xv]), 4),
                                     p=round(float(r.pvalues[xv]), 3), n=len(F)))
            # AI index gets the SAME falsification: it is prevalent and may proxy sector/verbosity
            if "ai_idx" in D.columns:
                Fa = F.merge(D.groupby("cik")["ai_idx"].mean().rename("ai").reset_index(), on="cik", how="left").dropna(subset=["ai"])
                if len(Fa) > 40:
                    Xa = Fa[["ai"] + base].astype(float)
                    if "ind" in Fa.columns and Fa["ind"].nunique() > 1:
                        Xa = pd.concat([Xa, pd.get_dummies(Fa["ind"].astype(str), prefix="i",
                                                           drop_first=True, dtype=float)], axis=1)
                    ra = sm.OLS(Fa[dep].values, sm.add_constant(Xa)).fit(cov_type="HC1")
                    rows.append(dict(outcome=dep, regressor="AI index (different construct)",
                                     beta=round(float(ra.params["ai"]), 4),
                                     p=round(float(ra.pvalues["ai"]), 3), n=len(Fa)))
            FALSI = pd.DataFrame(rows)
            print("PLACEBO FALSIFICATION of the firm-level results:")
            print(FALSI.to_string(index=False))
            bad = []
            for dep in ["vol", "worst"]:
                pr = FALSI[(FALSI.outcome == dep) & (FALSI.regressor == "PLACEBO lexicon")]
                rr = FALSI[(FALSI.outcome == dep) & (FALSI.regressor.str.contains("construct"))]
                if len(pr) and len(rr) and float(pr.p.iloc[0]) < 0.10:
                    bad.append(dep)
            if bad:
                print(f"\n  ⚠ PLACEBO ALSO SIGNIFICANT for {bad} → the firm-level result is a")
                print("    document/firm property (size, complexity, verbosity), NOT EA governance.")
                print("    Do not report the adaptability finding as a construct effect.")
            else:
                print("\n  ✓ placebo null while the construct is not → the firm-level result survives falsification.")
            globals()["FALSI"] = FALSI
            try: FALSI.to_csv(f"{OUT_DIR}/placebo_falsification.csv", index=False)
            except Exception: pass
        # is the firm-mean index just size / verbosity?
        if len(F) > 40 and "size" in F.columns:
            r_sz = float(F["real"].corr(F["size"])); r_pl = float(F["real"].corr(F["plac"]))
            print(f"\n  corr(firm-mean {idxc}, firm-mean size) = {r_sz:+.3f}"
                  f" · corr(construct, placebo) = {r_pl:+.3f}")
            # These correlations discriminate where the placebo REGRESSION cannot: if size is a
            # control, it absorbs the shared variance and the placebo can look null even when both
            # measures are the same document property.
            flags = []
            if abs(r_sz) > 0.5: flags.append(f"index is largely FIRM SIZE (|r|={abs(r_sz):.2f})")
            if abs(r_pl) > 0.5: flags.append(f"construct and PLACEBO move together (|r|={abs(r_pl):.2f}) "
                                             "⇒ both are measuring disclosure volume, not content")
            if flags:
                print("  ⚠ CONSTRUCT-VALIDITY FAILURE: " + "; ".join(flags))
                print("    Firm-level results using this index are not interpretable as EA governance.")
            else:
                print("  ✓ index is not collinear with size, and is distinct from the placebo.")
            globals()["FALSI_CORR"] = dict(corr_size=round(r_sz, 3), corr_placebo=round(r_pl, 3),
                                           flagged=bool(flags))

    # ---------- (b) SPECIFICATION CURVE ----------
    def curve(xcol, label):
        out = []
        W = D.dropna(subset=[xcol]).sort_values(["cik", "year"]).copy()
        lg = W[["cik", "year", xcol]].copy(); lg["year"] += 1
        W = W.merge(lg.rename(columns={xcol: "x_L"}), on=["cik", "year"], how="left")
        for c in ["size", "lev", "rnd"]:
            if c in W.columns:
                l2 = W[["cik", "year", c]].copy(); l2["year"] += 1
                W = W.merge(l2.rename(columns={c: c + "_L"}), on=["cik", "year"], how="left")
        ctl_all = [c + "_L" for c in ["size", "lev", "rnd"] if c + "_L" in W.columns]
        for dep in [d_ for d_ in ["roa", "q", "sga"] if d_ in W.columns]:
            for est in ["between", "within", "pooled"]:
                for use_ctl in [True, False]:
                    for use_ind in [True, False]:
                        S = W.dropna(subset=["x_L", dep] + (ctl_all if use_ctl else []))
                        if len(S) < 80: continue
                        S = S.copy()
                        if est == "between":
                            S["xx"] = S.groupby("cik")["x_L"].transform("mean")
                        elif est == "within":
                            S["xx"] = S["x_L"] - S.groupby("cik")["x_L"].transform("mean")
                        else:
                            S["xx"] = S["x_L"]
                        X = S[["xx"] + (ctl_all if use_ctl else [])].astype(float)
                        X = pd.concat([X, pd.get_dummies(S["year"].astype(str), prefix="y",
                                                         drop_first=True, dtype=float)], axis=1)
                        if use_ind and "industry_raw" in S.columns and S["industry_raw"].nunique() > 1:
                            X = pd.concat([X, pd.get_dummies(S["industry_raw"].astype(str), prefix="i",
                                                            drop_first=True, dtype=float)], axis=1)
                        try:
                            # standardise the OUTCOME: coefficients across roa / q / sga are
                            # otherwise on different scales and the pooled range is meaningless.
                            yv = S[dep].astype(float)
                            yz = (yv - yv.mean()) / (yv.std() or 1)
                            r = sm.OLS(yz.values, sm.add_constant(X)).fit(
                                cov_type="cluster", cov_kwds={"groups": S["cik"].values})
                            out.append(dict(index=label, outcome=dep, estimator=est,
                                            controls=use_ctl, industry=use_ind,
                                            beta=float(r.params["xx"]), p=float(r.pvalues["xx"]), n=len(S)))
                        except Exception:
                            continue
        return out
    allspecs = []
    _pairs = [("ci", "CI (Paper 1)")] + ([(idxc, "EAGI (Paper 2)")] if idxc not in (None, "ci") else [])
    for xc, lb in [(c_, l_) for c_, l_ in _pairs if c_ in D.columns]:
        if xc in D.columns: allspecs += curve(xc, lb)
    if allspecs:
        SPECCURVE = pd.DataFrame(allspecs)
        print("\nSPECIFICATION CURVE — distribution of the coefficient across reasonable specs:")
        print("  (β are on a STANDARDISED-OUTCOME scale: sd of the dependent variable per unit x)")
        for lb, gsub in SPECCURVE.groupby("index"):
            pos = (gsub.beta > 0)
            print(f"  {lb}: {len(gsub)} specs · median β={gsub.beta.median():+.4f} · "
                  f"range [{gsub.beta.min():+.4f}, {gsub.beta.max():+.4f}] · "
                  f"{(gsub.p < 0.05).mean():.0%} significant · {pos.mean():.0%} positive · "
                  + ("SIGN UNSTABLE" if 0.15 < pos.mean() < 0.85 else "sign stable"))
            for dep, gd in gsub.groupby("outcome"):
                print(f"      {dep:4s}: median β={gd.beta.median():+.4f} "
                      f"[{gd.beta.min():+.4f}, {gd.beta.max():+.4f}] · "
                      f"{(gd.p < 0.05).mean():.0%} sig · {(gd.beta > 0).mean():.0%} positive")
        # PRE-SPECIFIED PRIMARY: adherence/governance is a firm CHARACTERISTIC, so the
        # theoretically primary estimator is BETWEEN-firm with industry and year FE and controls.
        # Declaring it in advance is what separates a specification curve from spec-shopping.
        prim = SPECCURVE[(SPECCURVE.estimator == "between") & (SPECCURVE.controls) &
                         (SPECCURVE.industry) & (SPECCURVE.outcome == "roa")]
        if len(prim):
            print("\n  PRE-SPECIFIED PRIMARY SPEC (between-firm, ind+year FE, controls, ROA):")
            for _, r_ in prim.iterrows():
                print(f"    {r_['index']}: β={r_['beta']:+.4f} p={r_['p']:.3f} n={int(r_['n'])}")
            globals()["PRIMARY_SPEC"] = prim
        nsig = int((SPECCURVE.p < 0.05).sum()); ntot = len(SPECCURVE)
        print(f"\n  multiple-testing context: {nsig}/{ntot} specs significant at 5%; "
              f"~{0.05*ntot:.0f} expected by chance alone if no effect exists.")
        globals()["SPECCURVE"] = SPECCURVE
        try: SPECCURVE.to_csv(f"{OUT_DIR}/specification_curve.csv", index=False)
        except Exception: pass


In [ ]:
# ============ 10 · EXPORT + CIAOF SUMMARY ============
if G("df") is None or G("results") is None:
    print("SKIPPED — nothing to summarize yet. See RUN DIAGNOSTIC below.")
else:
    rows = []
    for y, (r1, r2) in results.items():
        rows.append(dict(outcome=y, model="H1", beta=r1.params["eagi_l"], se=r1.std_errors["eagi_l"],
                         p=r1.pvalues["eagi_l"], n=int(r1.nobs)))
        rows.append(dict(outcome=y, model="H2_ixn", beta=r2.params["ci_x_eagi_l"], se=r2.std_errors["ci_x_eagi_l"],
                         p=r2.pvalues["ci_x_eagi_l"], n=int(r2.nobs)))
    tab = pd.DataFrame(rows)
    tab.to_csv(f"{OUT_DIR}/eagi_main_results.csv", index=False)
    loto.to_csv(f"{OUT_DIR}/eagi_loto.csv", index=False)
    df.to_csv(f"{OUT_DIR}/panel_with_eagi.csv", index=False)
    try:
        _num = df.select_dtypes("number").round(8)
        PANEL_HASH = int(pd.util.hash_pandas_object(_num, index=False).sum() % (10 ** 12))
        df.to_parquet(f"{OUT_DIR}/panel_frozen.parquet", index=False)
        globals()["PANEL_HASH"] = PANEL_HASH
        print(f"frozen panel → panel_frozen.parquet · hash {PANEL_HASH}")
    except Exception as e:
        print("panel freeze skipped:", type(e).__name__)
    try:
        with pd.ExcelWriter(f"{OUT_DIR}/eagi_results.xlsx", engine="openpyxl") as xw:
            tab.to_excel(xw, sheet_name="main_results", index=False)
            loto.to_excel(xw, sheet_name="leave_one_term_out", index=False)
            df.to_excel(xw, sheet_name="panel_with_eagi", index=False)
            if G("VALIDITY_TBL") is not None: VALIDITY_TBL.to_excel(xw, sheet_name="convergent_validity", index=False)
            if G("INSTR_TABLE") is not None: INSTR_TABLE.to_excel(xw, sheet_name="instrument_variants", index=False)
        print("workbook → eagi_results.xlsx")
    except Exception as e:
        print("xlsx export skipped:", type(e).__name__)

    _key = "roa" if "roa" in results else (list(results)[0] if results else None)
    _estn = int(results[_key][0].nobs) if _key else 0
    _base = float(results[_key][0].params["eagi_l"]) if _key else 0.0
    _se0 = float(results[_key][0].std_errors["eagi_l"]) if _key else float("nan")
    _rng = float(loto.beta_roa.max() - loto.beta_roa.min()) if len(loto) else float("nan")
    _alt = G("ALT_NOTE") or []
    P1 = [x.replace("[P1] ", "").replace("[P1]", "") for x in _alt if x.startswith("[P1]")]
    P2 = [x.replace("[P2] ", "").replace("[P2]", "") for x in _alt if x.startswith("[P2]")]

    L = []; A = L.append
    A("=" * 72)
    A("RUN SUMMARY — ONE PIPELINE, TWO PAPERS")
    A(f"generated {pd.Timestamp.now():%Y-%m-%d %H:%M}")
    A(f"shared sample: {panel.shape[0]} firm-years · {panel.cik.nunique()} firms · "
      f"{int(panel.year.min())}–{int(panel.year.max())} · panel {G('PANEL_PATH') or 'DERIVED from source tables'}")
    A(f"fiscal-year alignment: {G('FY_ALIGN_NOTE') or 'n/a'}")
    A(f"panel hash (reproducibility): {G('PANEL_HASH') or 'n/a'} · frozen copy panel_frozen.parquet")
    A("REPRODUCIBILITY: seed=" + str(G("SEED")) + " · deterministic rule classifier is PRIMARY · "
      "LLM second classifier " + str(G("LLM_MODE") or "not run"))
    A("  LLM labels are cached to llm_techcom_labels.csv and ship with the replication package,")
    A("  so reruns are identical whether or not the API is reachable.")
    if G("KAPPA_RULES_LLM") is not None:
        A(f"  classifier agreement rules vs LLM (no human input): κ={G('KAPPA_RULES_LLM'):.3f}")
    A("")
    A("#" * 72)
    A("DECLARED ANALYSIS HIERARCHY (fixed before estimation; multiplicity is the main threat")
    A("to a programme this size — every result is defensible alone, the SET needs a structure)")
    A("#" * 72)
    A("  PRIMARY   (1 test per paper · 5% threshold, no adjustment needed)")
    A("    P1: CI → ROA, between-firm, industry+year FE, controls")
    A("    P2: measurement-paper tier, not exercised in this branch")
    A("  SECONDARY (pre-specified · Bonferroni within family)")
    A("    P1: within-firm estimate · inverted-U shape · dimension-level · P1-H3 dispersion")
    A("    P2: H1 levels (3 outcomes → p<0.0167) · H2 moderation · H3 variance battery")
    A("  EXPLORATORY (hypothesis-generating; report effect sizes, not significance verdicts)")
    A("    P1-H3b directional · P1-H3c quintiles · P1-H3d fit-to-potential · composite structure")
    A("    compliance channel · AI control and probe · ERP signals · long horizon · pre-shock")
    A("  Anything in the exploratory tier that survives should be RE-TESTED on new data, not")
    A("  promoted within this one.")
    A("")
    A("#" * 72)
    A("PAPER 1 — CIAOF: Core Integrity adherence and firm outcomes")
    A("#" * 72)
    A(f"Construct: {(G('DERIV_HOW') or {}).get('ci', 'n/a')}")
    A("Claim: firms with higher Core Integrity adherence achieve better outcomes;")
    A("originally specified as an inverted-U (an interior optimum in adherence).")
    A("-" * 72)
    for ln in P1: A("  " + ln)
    if not P1: A("  (alternative-design cell did not run — no between/within decomposition available)")
    if G("STRUCT_NOTE"):
        A("Structure of the composite — weakest link · complementarity · importance · necessity:")
        for ln in STRUCT_NOTE: A("  " + ln)
    if G("H3_NOTE"):
        A("P1-H3 — allocation, not level (does SELECTIVE adherence beat UNIFORM adherence?):")
        for ln in H3_NOTE: A("  " + ln)
    A("-" * 72)
    A("Paper 1 verdict (pre-publication — specify BOTH estimators up front, not as a revision):")
    A("  Theorise adherence as a firm CHARACTERISTIC rather than a decision margin. Report the")
    A("  within-firm estimate (clean identification, low power on a sticky construct) AND the")
    A("  between-firm estimate (powered, confounded) as answering two different questions:")
    A("    within  → does BECOMING more disciplined change outcomes?")
    A("    between → do disciplined FIRMS perform differently?")
    A("  The inverted-U is a shape claim and is therefore inherently cross-sectional; report it only")
    A("  if concave AND the vertex lies inside the observed range (checked above).")
    A("  Robustness: ccai_pca alternative composite (CI α is modest — show both).")
    A("  Identification roadmap for the follow-up: firm-level ERP install-base exposure from an")
    A("  EXTERNAL source (vendor-internal customer data would recreate the conflict CIAOF removed).")
    A("")
    A("#" * 72)
    A("(measurement analyses are not part of this branch)") if False else None
    A("#" * 72)
    if G("LEX_ALPHA") is not None:
        A(f"Instrument: α={G('LEX_ALPHA'):.3f} · mean inter-item r={G('LEX_MEAN_R'):.3f} · PC1 share={G('LEX_PC1'):.1%}")
        A(f"  independence benchmark for a {len(LEXICON)}-term mean = {1/np.sqrt(len(LEXICON)):.3f}; observed sd = {df.eagi.std():.3f}")
        if G("LEX_DEAD"): A(f"  terms in <5% of filings: {G('LEX_DEAD')}")
    if G("PREV_TBL") is not None:
        A("Term prevalence — the root cause (filing level vs firm-ever level):")
        for ln in PREV_TBL.round(3).to_string(index=False).splitlines()[:8]: A("  " + ln)
    if G("INSTR_TABLE") is not None:
        A("Instrument variants incl. FIRM-LEVEL aggregation (the CIAOF analogue for measurement):")
        for ln in INSTR_TABLE.to_string(index=False).splitlines(): A("  " + ln)
    if G("I106_NOTE"):
        A("Item 106 staggered-deadline design (feasibility-gated):")
        for ln in I106_NOTE: A("  " + ln)
    if G("CUST_NOTE"):
        A("Customisation VOLUME (XBRL capitalised software) — the 2x2's missing variable:")
        for ln in CUST_NOTE: A("  " + ln)
    if G("META_NOTE"):
        A("Filing-metadata instruments (filing lag · NT 10-K · amendments · SIC · filer status):")
        for ln in META_NOTE: A("  " + ln)
    if G("MAND_NOTE"):
        A("Mandatory-disclosure instruments (audited ITGC weakness · Item 106 cyber governance):")
        for ln in MAND_NOTE: A("  " + ln)
    if G("PROX_NOTE"):
        A("Proximity lexicon (the middle layer between exact phrases and raw roots):")
        for ln in PROX_NOTE: A("  " + ln)
    if G("CEIL_NOTE"):
        A("Lexicon ceiling — deterministic bound on any keyword method:")
        for ln in CEIL_NOTE: A("  " + ln)
    if G("SEM_NOTE"):
        A("Measurement of last resort — semantic scoring and alternative corpus:")
        for ln in SEM_NOTE: A("  " + ln)
    A("-" * 72)
    A("H1 — within-firm FE(firm+year), firm-clustered SE · dep ~ EAGI(t−1) + controls")
    for y in [k for k in ["roa", "q", "sga"] if k in results]:
        r1 = results[y][0]
        A(f"  {y.upper():4s} β={r1.params['eagi_l']:+.4f}  se={r1.std_errors['eagi_l']:.4f}  p={r1.pvalues['eagi_l']:.3f}"
          f"  per-1sd Δ={r1.params['eagi_l'] * (G('SD_EAGI_L') or 0):+.4f}"
          + (f"  | DK p={RESULTS_DK[y][0].pvalues['eagi_l']:.3f}" if y in (G("RESULTS_DK") or {}) else ""))
    A("H2 — centred interaction CI(t−1) × EAGI(t−1)")
    for y in [k for k in ["roa", "q", "sga"] if k in results]:
        r2 = results[y][1]
        A(f"  {y.upper():4s} β_ixn={r2.params['ci_x_eagi_l']:+.4f}  se={r2.std_errors['ci_x_eagi_l']:.4f}"
          f"  p={r2.pvalues['ci_x_eagi_l']:.3f}  per-1sd×1sd Δ="
          f"{r2.params['ci_x_eagi_l'] * (G('SD_EAGI_L') or 0) * (G('SD_CI_L') or 0):+.4f}")
    for _y in [k for k in ["roa", "q", "sga"] if k in results and k in (G("RESULTS_DK") or {})]:
        _sc, _sd2 = results[_y][0].std_errors["eagi_l"], RESULTS_DK[_y][0].std_errors["eagi_l"]
        if _sd2 > 0 and max(_sc / _sd2, _sd2 / _sc) > 2:
            A(f"  ⚠ {_y.upper()}: clustered vs DK SEs differ {max(_sc/_sd2, _sd2/_sc):.1f}× — T≈7 makes DK unreliable; use clustered.")
    A("Better-matched designs (sticky construct → between-firm; theory → adaptability):")
    for ln in P2: A("  " + ln)
    if G("AI_NOTE"):
        A("AI positive control + Paper 3 probe:")
        for ln in AI_NOTE: A("  " + ln)
    if G("AIFALS") is not None:
        A("Falsification of the AI results:")
        for ln in AIFALS.to_string(index=False).splitlines(): A("  " + ln)
    if G("VAR_NOTE"):
        A("Variance battery · pre-shock design · 2x2 heterogeneity:")
        for ln in VAR_NOTE: A("  " + ln)
    if G("H2S_NOTE"):
        A("H2 re-tested with a NON-TEXT moderator + formative-construct dimension analysis:")
        for ln in H2S_NOTE: A("  " + ln)
    if G("COMP_NOTE"):
        A("Compliance channel — is the visible vocabulary the meaningful one?")
        for ln in COMP_NOTE: A("  " + ln)
    if G("FALSI") is not None:
        A("Placebo falsification of the firm-level results:")
        for ln in FALSI.to_string(index=False).splitlines(): A("  " + ln)
    if G("FALSI_CORR") is not None:
        A(f"Construct-validity correlations: corr(index, size)={FALSI_CORR['corr_size']:+.3f} · "
          f"corr(index, placebo)={FALSI_CORR['corr_placebo']:+.3f}"
          + ("  ⚠ FLAGGED — firm-level results not interpretable as the construct"
             if FALSI_CORR["flagged"] else "  ✓ distinct from size and placebo"))
    if G("PRIMARY_SPEC") is not None:
        A("PRE-SPECIFIED PRIMARY SPEC (between-firm, industry+year FE, controls, ROA):")
        for _, r_ in PRIMARY_SPEC.iterrows():
            A(f"  {r_['index']}: β={r_['beta']:+.4f} p={r_['p']:.3f} n={int(r_['n'])}")
    if G("SPECCURVE") is not None:
        A("Specification curve (all reasonable specs; fragile results must be shown as a distribution):")
        for lb, gs_ in SPECCURVE.groupby("index"):
            A(f"  {lb}: {len(gs_)} specs · median β={gs_.beta.median():+.4f} · "
              f"range [{gs_.beta.min():+.4f}, {gs_.beta.max():+.4f}] · "
              f"{(gs_.p < 0.05).mean():.0%} significant · {(gs_.beta > 0).mean():.0%} positive")
    A("Robustness (ROA):")
    if len(loto):
        A(f"  leave-one-term-out β ∈ [{loto.beta_roa.min():+.4f}, {loto.beta_roa.max():+.4f}]")
    A(f"  winsorized β={beta_winsor:+.4f} · t−2 β={beta_lag2:+.4f} · placebo β="
      + ("n/a" if beta_placebo is None else f"{beta_placebo:+.4f}"))
    if G("VALIDITY_TBL") is not None:
        A("Convergent validity (within_firm = the variation FE exploits):")
        for ln in VALIDITY_TBL.to_string(index=False).splitlines(): A("  " + ln)
        VALIDITY_TBL.to_csv(f"{OUT_DIR}/eagi_convergent_validity.csv", index=False)
    if _key:
        _mde1 = 2.802 * _se0 * (G("SD_EAGI_L") or 0)
        A(f"Power: smallest detectable per-1sd {_key.upper()} effect = {_mde1:+.4f} (80% power)")
        _rel = G("LEX_ALPHA")
        if _rel is not None and _rel == _rel:
            A(f"  attenuation-corrected at reliability {float(_rel):.3f}: {_mde1/max(float(_rel),0.01):+.4f}")
            if float(_rel) < 0.3:
                A("  ⇒ NULL IS UNINFORMATIVE ABOUT THE ECONOMICS — predicted by the instrument alone.")
    if G("TC_BOUNDS") is not None:
        _lo, _hi, _ns, _sig = TC_BOUNDS
        A(f"TechCom extreme-coding bounds on β(→ROA): [{_lo:+.4f}, {_hi:+.4f}] · ambiguous share "
          f"{(G('TC_REVIEW_SHARE') or 0):.1%}")
        A("  " + ("null at both extremes → ambiguous cases cannot change the conclusion" if _ns
                  else ("robust at both extremes" if _sig
                        else "conclusion flips inside the bounds → coding of ambiguous cases is decisive")))
    if G("TC_SENS") is not None:
        A("TechCom coding scenarios (raw regex · lower bound · upper bound):")
        for ln in TC_SENS.to_string(index=False).splitlines(): A("  " + ln)
    if G("CS_NOTE"):
        A("Callaway-Sant'Anna (hand-rolled, not-yet-treated controls):")
        for ln in CS_NOTE: A("  " + ln)
    A("Event study — board TechCom formation:")
    for ln in (G("es_note") or "not estimated").splitlines(): A("  " + ln)
    if G("LEXVAL_MIX"): A(f"Lexicon context auto-labels: {G('LEXVAL_MIX')} (human coding sheet in OUT_DIR)")
    A("-" * 72)
    A("Paper 2 verdict:")
    A("  Report as a MEASUREMENT paper: a pre-registered disclosure index, a documented null, and")
    A("  quantified evidence on whether 10-K language can carry the construct at all. Structural")
    A("  measures (board technology committee, named CIO/CTO/Chief Architect) are the bridge to")
    A("  a follow-up effects paper; repository telemetry serves as a tech-subsample validation.")
    A("")
    if G("AGREE_TBL") is not None:
        A("Coding agreement (machine vs human gold standard):")
        for ln in AGREE_TBL.to_string(index=False).splitlines(): A("  " + ln)
    A("=" * 72)
    A("SHARED ARTEFACTS")
    A(f"  results: eagi_main_results.csv · eagi_loto.csv · panel_with_eagi.csv · eagi_results.xlsx")
    A(f"  validation: techcom_evidence.csv · techcom_verified.csv · exec_tech_roles.csv · lexicon_validation_sheet.xlsx")
    A(f"  instrument: eagi_instrument_variants.csv   →  all in {OUT_DIR}")
    A("  Tool linkage: TRVA Navigator operationalises Paper 1 (CIAOF dimensions → archetype gating)")
    A("  and consumes Paper 2's governance framing (governed capacity to choose deviations).")
    A("  Empirical results require NO human coding: rules are primary, LLM labels are cached data.")
    A("  Human coding is a QUALITATIVE FOLLOW-UP study (gold standard files are emitted for it).")
    A("  (legacy note) Human tasks reduced to a GOLD STANDARD only: code techcom_goldstandard.csv (~45 rows) and a")
    A("  slice of lexicon_validation_sheet.xlsx, save with a '_coded' suffix, re-run → κ is computed")
    A("  automatically and machine labels are then defensible for the full corpus.")
    A("Inference notes: SEs firm-clustered; H1 tests 3 outcomes → Bonferroni 5% is p<0.0167;")
    A("  H2 main effects centred; capex unavailable → capitalised-software intensity substituted.")
    A("=" * 72)
    globals()["RUN_SUMMARY_TEXT"] = "\n".join(L)
    open(f"{OUT_DIR}/ciaof_run_summary.txt", "w").write(RUN_SUMMARY_TEXT)
    print("\n" + RUN_SUMMARY_TEXT)


In [ ]:
# ============ 10b · PAPER NUMBERS EXPORT — the authoritative reconciliation artefact ============
# Every bracketed figure in either working paper must be reconciled against a single archived
# source. This cell IS that source. It harvests values from the globals the run already produced
# (never recomputing, so it cannot diverge from what was estimated), writes them keyed and
# machine-readable, renders the figures, and emits a checklist ordered to match Appendix D.2.
PAPER_ROWS, FIGS = [], []
if G("df") is None or G("OUT_DIR") is None:
    print("SKIPPED - run did not reach estimation.")
else:
    import datetime, math
    def _rec(paper, section, key, label, value=None, se=None, p=None, n=None, note=""):
        PAPER_ROWS.append(dict(paper=paper, section=section, key=key, label=label,
                               value=value, se=se, p=p, n=n, note=note))
    def _f(x, nd=6):
        try:
            v = float(x)
            return None if (v != v or math.isinf(v)) else round(v, nd)
        except Exception:
            return None
    # ---- parse the note blocks the analysis cells produced, so nothing is retyped by hand ----
    _num = r"([-+]?\d+\.?\d*(?:[eE][-+]?\d+)?)"
    def _harvest(notes, paper, section):
        for ln in (notes or []):
            t = str(ln).strip()
            if not t: continue
            b = re.search(r"(?:beta|\u03b2|b1|b2)\s*[_a-zA-Z]*\s*=\s*" + _num, t)
            pv = re.search(r"\bp\s*=\s*" + _num, t)
            nn = re.search(r"\bn\s*=\s*(\d+)", t)
            if b or pv:
                lab = re.split(r"(?:beta|\u03b2|b1|b2)\s*[_a-zA-Z]*\s*=", t)[0].strip(" :-\u2014")[:110] or t[:110]
                _rec(paper, section, "", lab,
                     _f(b.group(1)) if b else None, None,
                     _f(pv.group(1), 4) if pv else None,
                     int(nn.group(1)) if nn else None)
    # ---------- PAPER 1 ----------
    _rec("P1", "sample", "n_firmyears", "estimation panel firm-years", len(df))
    _rec("P1", "sample", "n_firms", "distinct firms", int(df["cik"].nunique()))
    _rec("P1", "sample", "year_min", "first year", int(df["year"].min()))
    _rec("P1", "sample", "year_max", "last year", int(df["year"].max()))
    _rec("P1", "sample", "panel_hash", "frozen panel hash", G("PANEL_HASH"))
    if "ci" in df.columns:
        _b = _f(df.groupby("cik")["ci"].mean().std()); _w = _f(df.groupby("cik")["ci"].transform(lambda x: x - x.mean()).std())
        _rec("P1", "4.1", "ci_sd_between", "CI between-firm SD", _b)
        _rec("P1", "4.1", "ci_sd_within", "CI within-firm SD", _w)
        if _b and _w:
            _rec("P1", "4.1", "ci_between_share", "between share of CI variance",
                 _f(_b**2 / (_b**2 + _w**2), 4))
    if G("PRIMARY_SPEC") is not None:
        for _, r_ in PRIMARY_SPEC.iterrows():
            _rec("P1" if "CI" in str(r_["index"]) else "P2", "3.4/4.2", "primary_spec",
                 f"PRE-SPECIFIED PRIMARY - {r_['index']}", _f(r_["beta"]), None, _f(r_["p"], 4), int(r_["n"]))
    _harvest(G("H3_NOTE"), "P1", "4.4-4.6")
    _harvest(G("STRUCT_NOTE"), "P1", "4.7")
    _harvest([x for x in (G("ALT_NOTE") or []) if x.startswith("[P1]")], "P1", "4.2/4.8")
    if G("DIMS") is not None:
        for _, r_ in DIMS.iterrows():
            _rec("P1", "4.7", "dim_" + str(r_["dimension"]), "dimension-level between-firm",
                 _f(r_["beta_between"]), None, _f(r_["p"], 4), int(r_["n"]))
    # ---------- PAPER 2 ----------
    for k, lab in [("LEX_ALPHA", "EAGI Cronbach alpha"), ("LEX_MEAN_R", "mean inter-item r"),
                   ("LEX_PC1", "PC1 variance share")]:
        if G(k) is not None: _rec("P2", "4.1", k.lower(), lab, _f(G(k), 4))
    if G("LEX_DEAD") is not None:
        _rec("P2", "4.1", "dead_terms", "terms present in <5% of filings", len(G("LEX_DEAD")),
             note="; ".join(map(str, G("LEX_DEAD")))[:400])
    if "eagi" in df.columns:
        _rec("P2", "4.1", "eagi_sd", "observed SD of the 20-term index", _f(df["eagi"].std(), 4))
        _rec("P2", "4.1", "eagi_indep_benchmark", "1/sqrt(k) independence benchmark",
             _f(1/np.sqrt(len(LEXICON)), 4))
    for src, sec in [("CEIL_NOTE", "4.3"), ("PROX_NOTE", "4.3"), ("SEM_NOTE", "4.3"),
                     ("AI_NOTE", "4.2/5"), ("MAND_NOTE", "4.6"), ("META_NOTE", "4.6"),
                     ("COMP_NOTE", "4.7"), ("I106_NOTE", "4.6"), ("VAR_NOTE", "4.4"),
                     ("H2S_NOTE", "4.4"), ("CS_NOTE", "4.6"), ("CUST_NOTE", "6.4")]:
        _harvest(G(src), "P2", sec)
    _harvest([x for x in (G("ALT_NOTE") or []) if x.startswith("[P2]")], "P2", "4.4")
    if G("results") is not None:
        for y, (r1, r2) in results.items():
            _rec("P2", "4.4", f"H1_{y}", f"H1 within-firm FE - {y.upper()}",
                 _f(r1.params["eagi_l"]), _f(r1.std_errors["eagi_l"]), _f(r1.pvalues["eagi_l"], 4), int(r1.nobs))
            _rec("P2", "4.4", f"H2_{y}", f"H2 centred interaction - {y.upper()}",
                 _f(r2.params["ci_x_eagi_l"]), _f(r2.std_errors["ci_x_eagi_l"]),
                 _f(r2.pvalues["ci_x_eagi_l"], 4), int(r2.nobs))
    for nm, tbl in [("instrument_variants", G("INSTR_TABLE")), ("variance_battery", G("VARTBL")),
                    ("techcom_sensitivity", G("TC_SENS")), ("placebo_falsification", G("FALSI")),
                    ("ai_falsification", G("AIFALS")), ("cs_att", G("CS_TBL")),
                    ("convergent_validity", G("VALIDITY_TBL")), ("lexicon_ceiling", G("CEIL_TBL")),
                    ("term_prevalence", G("PREV_TBL")), ("coding_agreement", G("AGREE_TBL"))]:
        if tbl is not None and len(tbl):
            try: tbl.to_csv(f"{OUT_DIR}/tbl_{nm}.csv", index=False)
            except Exception: pass
    if G("SPECCURVE") is not None:
        for lb, gs_ in SPECCURVE.groupby("index"):
            _rec("P1" if "CI" in str(lb) else "P2", "4.2", "speccurve",
                 f"specification curve - {lb}: {len(gs_)} specs, "
                 f"{(gs_.p<0.05).mean():.0%} significant, {(gs_.beta>0).mean():.0%} positive",
                 _f(gs_.beta.median()), None, None, len(gs_),
                 note=f"range [{gs_.beta.min():+.4f}, {gs_.beta.max():+.4f}]")
    # ---------- write ----------
    PN = pd.DataFrame(PAPER_ROWS)
    stamp = f"{pd.Timestamp.now():%Y-%m-%d %H:%M}"
    try:
        PN.to_csv(f"{OUT_DIR}/paper_numbers.csv", index=False)
        json.dump({r["key"] or f"{r['paper']}_{r['section']}_{i}": r for i, r in enumerate(PAPER_ROWS)},
                  open(f"{OUT_DIR}/paper_numbers.json", "w"), indent=1, default=str)
    except Exception as e:
        print("csv/json export failed:", type(e).__name__)
    L2 = []; A2 = L2.append
    A2(f"# Paper numbers - authoritative export\n")
    A2(f"generated {stamp} - notebook {G('NB_VERSION')} - panel hash {G('PANEL_HASH')}\n")
    A2("Every bracketed figure in either working paper reconciles to a row below. Values are")
    A2("harvested from the run's own outputs; nothing here is recomputed or retyped.\n")
    for pap, title in [("P1", "CIAOF")]:
        A2(f"\n## {title}\n")
        sub = PN[PN.paper == pap]
        for sec in sorted(sub.section.unique()):
            A2(f"\n### Section {sec}\n")
            A2("| statistic | value | se | p | n |")
            A2("|---|---|---|---|---|")
            for _, r_ in sub[sub.section == sec].iterrows():
                A2(f"| {str(r_['label'])[:100]} | {r_['value']} | {r_['se'] if r_['se'] is not None else ''} "
                   f"| {r_['p'] if r_['p'] is not None else ''} | {r_['n'] if r_['n'] is not None else ''} |")
    A2("\n\n## Verbatim run blocks (source of truth for anything not parsed above)\n")
    for nm in ["CEIL_NOTE", "PROX_NOTE", "SEM_NOTE", "AI_NOTE", "VAR_NOTE", "H2S_NOTE", "H3_NOTE",
               "STRUCT_NOTE", "COMP_NOTE", "I106_NOTE", "MAND_NOTE", "META_NOTE", "CUST_NOTE",
               "CS_NOTE", "ALT_NOTE"]:
        v = G(nm)
        if v:
            A2(f"\n### {nm}\n```")
            for ln in v: A2(str(ln))
            A2("```")
    if G("RUN_SUMMARY_TEXT"):
        A2("\n\n## Full run summary\n```")
        A2(RUN_SUMMARY_TEXT); A2("```")
    try:
        open(f"{OUT_DIR}/PAPER_NUMBERS.md", "w").write("\n".join(L2))
    except Exception as e:
        print("markdown export failed:", type(e).__name__)
    # ---------- figures ----------
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        FD = f"{OUT_DIR}/figures"; os.makedirs(FD, exist_ok=True)
        def _save(fig, name, caption):
            fig.tight_layout(); fig.savefig(f"{FD}/{name}.png", dpi=200); plt.close(fig)
            FIGS.append((name, caption))
        if G("SPECCURVE") is not None:
            for lb, gs_ in SPECCURVE.groupby("index"):
                g2 = gs_.sort_values("beta").reset_index(drop=True)
                fig, ax = plt.subplots(figsize=(7, 3.4))
                cols = ["#c0392b" if pp < .05 else "#9a9a9a" for pp in g2.p]
                ax.bar(range(len(g2)), g2.beta, color=cols)
                ax.axhline(0, color="k", lw=.8)
                ax.set_xlabel("specification (ordered by coefficient)")
                ax.set_ylabel("beta (standardised outcome)")
                ax.set_title(f"Specification curve - {lb}")
                _save(fig, f"fig_speccurve_{re.sub(r'[^A-Za-z0-9]+','_',str(lb))}",
                      "distribution of the coefficient across all defensible specifications; red = p<0.05")
        if G("PREV_TBL") is not None and len(PREV_TBL):
            t = PREV_TBL.sort_values("share_firms_ever").tail(20)
            fig, ax = plt.subplots(figsize=(7, 5))
            ax.barh(t["term"].astype(str), t["share_firms_ever"], color="#1a6b72")
            ax.axvline(.05, color="#c0392b", ls="--", lw=1, label="5% threshold")
            ax.set_xlabel("share of firms using the term at least once"); ax.legend()
            ax.set_title("Lexicon term prevalence")
            _save(fig, "fig_term_prevalence", "term prevalence at firm-ever level; the 5% line is the usability threshold")
        if G("INSTR_TABLE") is not None and len(INSTR_TABLE):
            t = INSTR_TABLE.dropna(subset=["spearman_brown"]).sort_values("spearman_brown")
            fig, ax = plt.subplots(figsize=(7, max(3, .32*len(t))))
            ax.barh(t["variant"].astype(str), t["spearman_brown"], color="#6b6457")
            ax.axvline(.5, color="#c0392b", ls="--", lw=1, label="SB = 0.50")
            ax.set_xlabel("Spearman-Brown"); ax.legend()
            ax.set_title("Instrument reliability across constructions")
            _save(fig, "fig_instrument_variants", "no construction reaches conventional reliability")
        if G("VARTBL") is not None and len(VARTBL):
            fig, ax = plt.subplots(figsize=(7, 3.6))
            lab = VARTBL["risk_measure"].astype(str) + " ~ " + VARTBL["regressor"].astype(str)
            ax.barh(lab, VARTBL["p"], color=["#1a7a4a" if pp < .05 else "#9a9a9a" for pp in VARTBL["p"]])
            ax.axvline(.05, color="#c0392b", ls="--", lw=1)
            ax.set_xlabel("p-value"); ax.set_title("Variance battery")
            _save(fig, "fig_variance_battery", "risk-measure results; significance alone is insufficient - see placebo columns")
        if G("CS_TBL") is not None and len(CS_TBL):
            t = CS_TBL.dropna(subset=["att"])
            fig, ax = plt.subplots(figsize=(6.5, 3.6))
            ax.errorbar(t["event_time"], t["att"],
                        yerr=[1.96*(s if s == s else 0) for s in t["se"].fillna(0)],
                        fmt="o-", color="#1a6b72", capsize=3)
            ax.axhline(0, color="k", lw=.8); ax.axvline(-1, color="#c8a84b", ls="--", lw=1)
            ax.set_xlabel("event time (reference tau = -1)"); ax.set_ylabel("ATT")
            ax.set_title("Callaway-Sant'Anna event-time ATT")
            _save(fig, "fig_cs_event_study", "not-yet-treated controls, multiplier bootstrap; pre-periods should be indistinguishable from zero")
        if G("DIMS") is not None and len(DIMS):
            t = DIMS.sort_values("beta_between")
            fig, ax = plt.subplots(figsize=(7, 3.2))
            ax.barh(t["dimension"].astype(str).str.replace("kpi_", "", regex=False),
                    t["beta_between"], color=["#c0392b" if b < 0 else "#1a7a4a" for b in t["beta_between"]])
            ax.axvline(0, color="k", lw=.8); ax.set_xlabel("between-firm beta")
            ax.set_title("CIAOF dimension-level associations")
            _save(fig, "fig_dimension_level", "the composite can average opposing dimension effects")
        if "ci" in df.columns and "roa" in df.columns:
            b_ = df.groupby("cik").agg(ci=("ci", "mean"), roa=("roa", "mean")).dropna()
            if len(b_) > 20:
                b_["q"] = pd.qcut(b_["ci"], 5, labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"],
                                  duplicates="drop")
                m_ = b_.groupby("q", observed=True)["roa"].mean()
                fig, ax = plt.subplots(figsize=(6, 3.2))
                ax.bar(m_.index.astype(str), m_.values, color="#c8a84b")
                ax.set_ylabel("mean ROA (raw)"); ax.set_xlabel("adherence quintile")
                ax.set_title("ROA by adherence quintile (descriptive)")
                _save(fig, "fig_quintiles_raw", "raw means without controls; the paper reports adjusted means")
        print(f"figures written: {len(FIGS)} -> {FD}")
        for nm, cap in FIGS: print(f"    {nm}.png  - {cap}")
        open(f"{FD}/CAPTIONS.md", "w").write(
            "# Figure captions\n\n" + "\n".join(f"- **{n}.png** - {c}" for n, c in FIGS))
    except Exception as e:
        print("figure export skipped:", type(e).__name__, str(e)[:90])
    globals()["PAPER_NUMBERS"] = PN
    print(f"\nPAPER NUMBERS EXPORT -> {OUT_DIR}")
    print(f"  paper_numbers.csv   {len(PN)} keyed statistics")
    print( "  paper_numbers.json  same, machine-readable")
    print( "  PAPER_NUMBERS.md    reconciliation document (use this for Appendix D.2)")
    print(f"  tbl_*.csv           {sum(1 for _ in os.listdir(OUT_DIR) if _.startswith('tbl_'))} result tables")
    print(f"  figures/            {len(FIGS)} figures + CAPTIONS.md")
    print(f"  panel hash          {G('PANEL_HASH')}")


In [ ]:
# ============ 11 · RUN DIAGNOSTIC — always executes; paste this block with any traceback ============
def G(n):
    return globals().get(n)
def _n(x):
    try: return len(x)
    except Exception: return None

prefetch = bool(G("PREFETCH_ONLY"))
stages = [
    ("cell 1 · discovery",   (G("PANEL_PATH") is not None) or prefetch or (G("DERIVED_PANEL") is not None)),
    ("cell 2 · panel load",  (G("panel") is not None) or prefetch),
    ("cell 3 · fetchers",    G("doc_text") is not None),
    ("cell 4 · eagi counts", G("counts") is not None),
    ("cell 4 · merge (df)",  (G("df") is not None) or prefetch),
    ("cell 5 · techcom",     G("tc") is not None),
    ("cell 6 · estimation",  (G("results") is not None) or prefetch),
    ("cell 7 · oster",       (G("oster_res") is not None) or prefetch),
    ("cell 8 · event study", (G("es_note") is not None) or prefetch),
    ("cell 9 · robustness",  (G("beta_winsor") is not None) or prefetch),
    ("cell 5b · validity",   (G("VALIDITY_TBL") is not None) or prefetch),
    ("cell 5c · validation", (G("VALID_ARTIFACTS") is not None) or prefetch),
    ("cell 4b · variants",   (G("INSTR_TABLE") is not None) or prefetch),
    ("cell 4d · ceiling",    (G("CEIL_NOTE") is not None) or prefetch),
    ("cell 4e · proximity",  (G("PROX_NOTE") is not None) or prefetch),
    ("cell 5e · mandatory",  (G("MAND_NOTE") is not None) or prefetch),
    ("cell 5f · filing meta", (G("META_NOTE") is not None) or prefetch),
    ("cell 5g · cust volume", (G("CUST_NOTE") is not None) or prefetch),
    ("cell 6i · item106 DiD", (G("I106_NOTE") is not None) or prefetch),
    ("cell 8c · CS estimator",(G("CS_NOTE") is not None) or prefetch),
    ("cell 4c · semantic",   (G("SEM_NOTE") is not None) or prefetch),
    ("cell 6b · alt designs",(G("ALT_NOTE") is not None) or prefetch),
    ("cell 5d · erp signals", (G("ERP_TBL") is not None) or prefetch),
    ("cell 8b · tc sensitivity", (G("TC_SENS") is not None) or prefetch),
    ("cell 6c · compliance",  (G("COMP_NOTE") is not None) or prefetch),
    ("cell 9b · falsification",(G("SPECCURVE") is not None) or prefetch),
    ("cell 6d · H2 structural",(G("H2S_NOTE") is not None) or prefetch),
    ("cell 6e · variance/shock",(G("VAR_NOTE") is not None) or prefetch),
    ("cell 6f · AI control",   (G("AI_NOTE") is not None) or prefetch),
    ("cell 6g · P1-H3 allocation",(G("H3_NOTE") is not None) or prefetch),
    ("cell 6h · composite structure",(G("STRUCT_NOTE") is not None) or prefetch),
    ("cell 9c · AI falsification",(G("AIFALS") is not None) or prefetch),
    ("cell 10b · paper numbers",(G("PAPER_NUMBERS") is not None) or prefetch),
]
print("=" * 60)
print("RUN DIAGNOSTIC")
print(f"notebook: {G('NB_VERSION') or 'UNKNOWN (pre-v7.1 file — Colab is running a cached notebook)'}")
print(f"mode: {'PREFETCH-ONLY' if prefetch else ('DERIVED' if G('DERIVED_PANEL') is not None else ('PANEL' if G('PANEL_PATH') else 'FAILED @ discovery'))}")
print(f"PANEL_PATH = {G('PANEL_PATH')}")
print(f"cik_registry_match = {G('CIK_SHARE')}")
_dl = G("DERIVE_LOG")
if _dl:
    print("--- derivation report ---")
    for _ln in _dl.strip().splitlines()[-30:]: print(_ln)
    print("--- end derivation report ---")
print(f"firms={_n(G('FIRM_CIKS'))} · panel_rows={_n(G('panel'))} · term_counts={_n(G('counts'))} · techcom={_n(G('tc'))} · merged_df={_n(G('df'))}")
first_fail = None
for name, ok in stages:
    print(("  ✓ " if ok else "  ✗ ") + name)
    if not ok and first_fail is None: first_fail = name
if first_fail:
    print(f"\nROOT CAUSE STAGE → {first_fail}")
    print("Scroll to that cell's red traceback and paste it TOGETHER with this block.")
elif prefetch:
    print("\nPREFETCH COMPLETE — text layer banked. Map the panel (inventory above / COLMAP), re-run to finish from cache.")
else:
    print("\nAll stages complete — paste the CIAOF RUN SUMMARY from cell 10.")

# ---- full results summary reprinted here so ONE final block carries everything ----
_summary = G("RUN_SUMMARY_TEXT")
if _summary:
    print("\n" + _summary)
else:
    print("\n(no results summary — the run did not reach estimation; see ROOT CAUSE STAGE above)")
